# CIC-DDoS2019 LightGBM CPU baseline

Smoke: 2,000 rows per source file and at most 10 new iterations in this session; target remains exactly 100. Resume/checkpoint state is synchronized with S3.


In [ ]:
from pathlib import Path
import base64
import hashlib
import json
import os
import subprocess
import sys
import time
import zlib

PROJECT_NAME = "Luan-Van-LightGBM-Parquet-Github-v2"
SESSION_MAXIMUM_HOURS = 12.0
SESSION_STOP_BEFORE_MINUTES = 30.0
os.environ["PIPELINE_SESSION_DEADLINE_EPOCH"] = str(
    time.time() + SESSION_MAXIMUM_HOURS * 3600.0 - SESSION_STOP_BEFORE_MINUTES * 60.0
)
os.environ.setdefault("MALLOC_ARENA_MAX", "2")
PROJECT_DIR = Path("/kaggle/working") / PROJECT_NAME
SOURCE_DIR = PROJECT_DIR / "source"
PREPARED_DIR = PROJECT_DIR / "prepared"
RUNS_DIR = PROJECT_DIR / "runs"
for directory in (SOURCE_DIR, PREPARED_DIR, RUNS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

encoded_files = json.loads("{\"checkpoint.py\": \"eNrtPWtv3EaS3wPkP3B5MEJmR5QdJ4vF5OZwsWMnxtqJYTl3t9AJBDXskbjikHN82Fa0+u9XVf1gv8jhSPLeHXABdj0ku6ur613VD4Vh+LzJ2sujNtuw4HVxcdn99OxN8Kyu2441wfqSra92dVF1bZBVefCBNcWmYHmQdfW2WAcnT4P2ulonYRh++cWXX2yaehuk6abv+oalaVBsd3XTQc+q7rKuqKsWW4m36/aD+n0JKJTFuXr+W1tX6qGsLy6K6kI916362V72XVGqx67YMvXQ90UuUMqzjuE3iZB8XlCP3+uKiYa7rEM8ZLu38Ci+dNc7wEF++KG6XgRvsh2+WwQn7L96Vq0ZTu7LL17/+tNPL94FK4l3csG61/CTNVGaVtkWCBPzljnbBH23Tqv6YxQHR/8StF2z/PKLAP5rGJCwUogm2ETimkCfOCnaelM326yLNGjtZfbNd39KN0XJIpzLkqawAD721VV6ft2xdhkANwG7J4+/+Tb4mv6xxs6LC9ZiE8GVhEPFcfDzx6K7JEIl9Y5VUdich3GQtdC6yksmYFDDS0AjOC/r9VWwXInvScOyPNLwibUew+hJv8O5R9Q7NmnCG1yyT/yXPv91VtVVsc7KFHEHElyXdZYvkV/WJIFfdQ6CvCJZS/J+u2tl+wV8bVGCs3ZdFKuXWdmCqLTA+PSKXber902Pz2yXNaAHTbuKwkW4CMJlGMcJBxyFfbc5+nNoom4RVOAQ++fCVSxF9FJEz5jNIsiheVGRVnEu0wR/AfmQbBwaJIApq7pke5UXTcQf5DTYp6Lt0vqKHgW6HUMxz5prII8OBlmftv1mU3yK9Pf8VfDHIEy67S7UBUWBEtLyMVxw2oNmrCSRvOKj+DKwhbfxs6eocpjV6hubUfEAUUjgx6YAyQr/swrdb5uyB7nR3tdtskETF8kGINNVHcWiCXxu2K7M1ixSMzV44zL0EuhdN9ecp+JhqWzIqbAqpyCqC+T02dmCSJEa+tx+0J5tzjuSU8IocqhYAycmIaHdR0wUDF1G1Eu/gIAnKXO0iGCUEEWc8xnAOj0T3+sGNGddNzlwN5CkGniD34HP+JE3s0xJsaHP4HywiTaa2cxEJQH6syqPoOehcrwIKvaxLCq2CsdMIopew8mV/Fisu3+nF5EU7AGN1fAztvtzCb4EQwpdR7429cdWsfwziblkrybjH7KyQMMtpXymgK/7BoUtRcSFSQMvZcu1+tqikECDiLP9NFRfwrPYLzZCpNinHVt3ZPdJKZqsumDREzQfXeRgEYPEPpE0AGnSEPjDSsHS+NtkRcuCf8vKnr1omrqJTEnbhAKbBFUw2PbgZtd11WWAJ/5bVH3dt0FfFUArfbQnSXLjYHf7fRDa8OvzljUfYH4oHaubAcTp8ruz2wBGKo23R98tz241KIKV6zJrW4juTgBZSX2I8SDcy/JshyLctxgODRag33ETzSPENYEHvtY7kJMhcuTvZbjIvdQGIsaiKro0jVpWbrBTtSkuloEjKKhx2XnJ8rQGcE2Rs2VwXtdl8HeSEuAp/mNLDVk4Agk+AxmPXSL+5jQUEEFuhtaIRiI+oK4OnUEGbBSCouWjM3BDHLbdxAbdgCoUHlw4DVP52cVpm32Cr11TMNQAeFKSKyBoDaC3O24Hfuc8a1nagn5UOULZwJgDALeJi8V5v75iGB+CVWDVh6IBjkKMGwGfFBzeJoXP0B9CozBO4HOxi2xgu4ahZ5gExtv4gIl/w2PdlTfsAoRsGiRvM40fMBtdB29q+Yw5Y2yysjzP1lfpnMEEgwRU8QMM2S+UnNgka4uLCgwPxMLrbtAPUw+GXqoDWWsXadDr9O27FyevfvrlxY/p819/efnqp/TtD+9/DscpY8K0qEMeEyOTyGwWc+c5L/7zz1kG7KgpwlnFZp9BXVMlqcgWE44UUUO8re5KNn3d26fiO0DwiqEglK4yaAj15z+sXGw9FHC8SvhWkUP2J8MaCLh5sdmwpg0ocwXmPvvt+V9evB9DTkxTISeeTeT4y3shJ+DayIHkvXz1HzZypqVxqORprLjlYG3pT7ouC3CklEeNqYykjfQCSJwIrYGOF6inesVHspNZiN7a7ALBh6JUcglKV/zOCSIMfTuwiMYZaOLnl/QPo8x411dYK+DsEChY5OVFiuRj1lSgiFH4qP0+gFQ7K42ST8O2GJpIH7kI/MAsf0kJmfTv/7prQOeb7npw95z8pFDkq4ENS5fygkvSvVqT7YxEQPXl1Znzuqufml/ZpzXbdcErakCEQZsDb+cRMSSQiIvy3R8vWRV4mAptJL1iLuMwiodecn4rjm4iqAJ2BegsPAaG/yvNMxhOjioKOig9pAKq/w2CU0yDRVTVsBIQ/AAhExgrqxjCAfLv5H34z6T0u1caeRPeaIJ/e3wje92GtmWhyEh+1rFs+krHEJ6KnPCbg+4wf32ygBaHo2Ek5/HV8VfxbRgboScFPAIBFFSRfRAOa+HAh/qRKaoQJEOIgwKyDF6QgKEAjFgUzEqyDgNmykZV4uEEdphyzJJ2QQKJZRR7RX5AbFTgh3kEK1dahVJK3P/ZwdgDkZO3zLAyMBZ6fh18E3z9dRBJwEcwbz8k21iB1j1qFUKP8uNHebCBzAa0MnrUxt8HNBqVa6vgUfJk04Yacxeyp0v7BZZ5WQQkiBNZq13wiYygRsXZtmRsF9nNIINiWL0eiAumAf2FKRfc2gytLOmERLDt2jFj6QqGMIGy4/BxrgX0WT8JzTCAKNBDWAY86UCmW3TtXsMnpHVATJvmECXm9ceKQjuukaDRgwEw8kEqFf3ddg2C5lbI6KX7B4xYpHiqtjwellig3NzcxvRyqAfJYgBkwG2XVWsWESzCKXbyBJr1TQggwmUgGoaYncvHW4dG9HqEQLus6VKeKAoatXXfrJmsDG6KKitTg25aiqxhBzn4bwQGYg6Qmm3fodNCjoI5XYOfR/3B0YIO3Ft/cRlkIoI7KottgTWUt7+evOe5/B3oT1Wa7KMI0TC1B2FGhfUxRJu15AlgsG0jm97U0IjbhzHipJmI0xXhMPAFL6JBAlfCiZygSbgNPQZPVG/YHLtN2ZEAuGcJxbSn4NZrMBOIWeQxBvZ/opTXMnYVPY7H2xmhhLI4cYKDRePd+CoKTPE07JsyPFtMN82zLlthvhqJTlTgxPLCno5YimxXNyH+CzoTaayQKwLx7R4YaMvqvls9+fPjxxNNVWjgkqjd1VWr7IUIG0JNLyVnQDzV79jntDmkhOxsClqQghHp+jaKRz08ltzv6ttnJA3K6rufLM/r76slfOAFyGRwVRVOWWYW9IWnFOi9wYcv4X/hCD84m0c+ety023CcoCJDsd5y+yTSFmRKsd6y7rLONUMMdgx41lc5jq5FfcrQGiGqEhowYhB0NNgJrL/8wC2Zx0UKESHT9048vQEQqEOaSwp/fv/+7QlJz/M6Z2BDVqvg28ffYmKKps92RTpY8uw6LIJA9R6tGoz2+CYEkPjll/qkX1/+hV3zh+4l0iG81R1VfU6BOC0WtT4fbpFIlNj0BNLvQC2OaQmG62MsCNyxSMX1hBqmZ9fQop7LuVI0YSdcEJrloFhg4YvKt+d5tnTtMnaQRpQ6ny0GA/fNY9ueunNTksAtT4qr0EJ07jTbvbbMa8dc2hqmFWeWcnkCeTOIwdPdRGsRPaOKzEqrziwCkNQV8nYOMvMMqqpNGGYgPoxo3NIOOiOq/nzR9pCY7gH0qC1+R1nFJQThVpGBVFdP8ZtGOqoLtf0WRVvbZcK7xQ+gmlbIZ0Z7+EqR4TBtpUAOYPdl57ECRjzNXU4wNpAsfWsAx+dmEV7r5LYbyoQbMKxaVsWF4ki4U5XCUvB8o9C8De/n91FTpoqFnkhgvOGUuRBZ5MqxaQ5Dw10PGoYTDAOqQPp56397jyBbBdpy+DmB9iHBticVTmAwac+HaYNRp5BZbg3QQ9jROHVmrKoGWQwTjacj31nxKg/XUSrRNu91giPOUOI23kF6A0VB2yUONJxwjBNxojmR2XPHqkAlDRX15Js0WhliVR0gefSaVRfdJRD/6EkcY7qJpnZCvriSvvrVWV+hgagIjXZcX2IXoXc4guieLX5+Ulyw7i4shW534CgSTDB0AHBGNRaWbcWmJEMn7sJdAD1fsMHm8s2MtJlIdsfNFOmac1bsa0RurPT9lRO8BZEhoHvMy/iuSA9Ad4ckLeYJB34vQTv5+YcjEJRDZG00iT0skT3AqU0ktLOTWjex9ea04NWLTbY+IK/dm9vOzm8npHvcCf9T8CxbX8HccXV2uwMGnhdl0V3z9YkSd+deHxVt2/MK7zbrjsC8YDmQlpfRF7XJ565ygU9SO4s+Q6VrzPGqMe/mfGcnfx4jaQ4/IRlGw/lCcZAbV/Dnu3Hbhe+dy34XbnBjjxvfO4lZ8xbu2+p5bxc+alWH3XMHunDcVpceJG4af6jzpEN2eIN6Mi6SgmkaXGCYoNqKPnLaUQPxnqd1A1//9PjxiCzP47ZBkVnMPjBOPShGfbj4dBLxQ4T6IWPSh41HD4hFD4tDD4lBP0/8OTWFeUL64DHnjHhzXqx51zjzwWPM/22LJPeMGmVw6IkBD6q2EDnLw4omSvpzVjLgvVAA/jTukPeH9ct5Gzue132ZU0GND6lVwgZfySu+pBpEL1+1bnDhuKBL67lDtQyPoBzd4PHABP/vWyqpf9KLaC6F/DSbYYNEdZcWL/fYHr22rXWb8Ly0L5UXYBeBUfs2CDARTr741DXZD80FLrCqFaZlcBNyIww/pTrfjq2x3ikem2fA7VDyzssABjniWfh6g8FT6S+FuzxTrhLX3KzAkfyrb9lO0PZwa7kJ3+sho8dA8sqwMd1bn9GcQXwKqeQCzHzJ1bpNSK6HUeONkYNKfSfaPYehT0gdQJr5COHS1IsQly+XpnpMbR6Q7PsR3MIa9xKuwue/vv1reD9dODT21CPOO+vA1KrGnLhxWva1+PJzyP1LGRyMyry2GvJ/IUj4nwoN/GHBhHsTEiZigHvY2Rkxwd54wMiY0bUTWexRh7VVucYv1iqN/QieI9QLtbtRHCwTm9kfeOuC6mdul0Dkl57F+j27/V/C1ORuDKku7+QmTSCTiJdwUFqyhsk/XR4f32jMuz2+8SmOfwL3OFd+z7Plkps6np4dsAcsQN9xf8iMVef93BtZgrVWgCUWwW/vXnNDN2biptOBgwuUoUuJWWGAvlPFTKIfIn0+qMTjP7g9pwpt5N1qzH/MUo9xS8FY9j3n3oHZPtBYJjZM5ki44ZpV5QUwQqO8RKEV2xtwDsXdi7ffjwGJB34Ls+pjydCor8qiuooMr4GLIHR7gm8Hm/fI5bRfMA6K7bX5xm7Ef9CGN7FvW+zB9m3XvuvuNkXLg7a4uYZj/ga3Q3elqfbE8di7f9PQEKwHTm5MGxrM3ZcmUFEnW9tIDn4aPqvz6/CM35sTQxxmXC9jnNd/ro7uvcmq7ILJ81LvXpz89uaFPNj78tXrFycwJ41vIR6Z4scmE86rINzCMCVuj8q2rXrZZttdydJtVhUbLP3J98apgcRmd4iJAAhijeZC9Smzcxhgyw91qLcACK9vMZDhkGLvlQHm2U5tUBAWXCtr6roj5QVNpRBvaMFniPEzjwetvW5EyHT8MgJt5Kdzm825v0BO13ONATcgw8SgB52y1l45VwaoSeKNAerBauVMGIMz3KvvfLDht0+hqbgoIlKEWPgma3fNKKFOCTpdiuJM7zgIRSMQUC4Mpp1u65J/TItcnXMkK8JyyXX7agjzHKGI0XiPEfsgPuvquq07ZRfap4npMfCNfibRmUQcK4MOTaWXoGOSjtkXY+HBYP5T5NJk0kKxQxv1L2tA8EiF+qoSP3dZ39LB3ZAW9skYcg0Lb/2zpSMzNM5pyOlqnJCXaJu8G3O1fA+Ap/0drgHg6fHe0/+AILX8TFSyKEVDeQllnZIdNO82vTFudaPLAzb4GH316K+Pto/yo0c/P3qDJ1Xto7KYZDlHZUmm35p3MOgnekx94h11wC04K3wrfKTnKC7/Is/Ezrqph3YqKL2+MWkoybUUw1juXDJtKQa2PzsYhMuR23vsnnxFC7daQxd18Z7WSjuK596+xqez8MmzR0EGvfZtcoev7rZvC+gimGlLvAUP3Gq8ZcRVNiI09nEVLmHqxK4QON5Lt90IUt4kIjuBpdYuDgjxuYP4CzWM4yCt9zidhgso4J17kITMrDgvLuTGGPPYN14cj1Dfk8HcCf5CI8dQPbJrHoMtVI3vdBMK9t5vAu8xRzqGTLYq6T51OL0JBluNp6d/IFJbPJC9bo/1+7IsdEQTQsVoNo0IpvYUDsmjpp7oeKwW6uA/cnRtZFabkEdHxzf8HOnYllWd5tQB50jnHceXbdV099dReDJDhfRrlRS2kRhVSPScHFw5QiXUMg7Yd4xubHBZhiWQnmja52dkorLPHlniqt2JJy5kdEGMiphpx7TjMAMyfHUE53Eqcimx/HHmPXCvNkT8KGgCBtHEWO2LyGvGM/Rt1q0vA69dGkVQn6yOonx/MJLGjXaHoTgIBGnUlK/qwF2yU7P2suCXRpovzzwlmX+4Z6Nij6sVcwo993QTM1wEjRDNkfKDr+sSvafHP9T8jF0pyW++E+LrBoVnnvoKNV5INI0oOIOYauDzeFlBj4z1RrRB2vPlnN+kzS/tdWzO0ivCWjtef6HbjG3IG5bRNdstoL3NvE26rMFalBmrL0yBl5G9VnXwKxtdFKjbBejd8Ks0KZnZsWYD6X+PsatRRpupfVqpgTcf0UDN8nPBdZtLiR4DftCClSNa4sCk4GzifI/iWdLr5iu+/Utk/oOVTR3Hn9G9uo7YJSTX1CgyivL8HQRUVb8dMFhN4WSupoiR5c7+P05YhT13y/JCsJzLjKmGxrW1zlICb7jQYMaurydvtTIkaNzHy07r9sN4H/gYOumj767nhYHEQodu+yJP7sxdA1jPlme+T+wkl8cauEMAvoYl3uh/cb5NxI3+obc5tbSieatdvQMlL34HCEu9SjngBVzL+3L0+76cXxlQ2rEjH+bk/s47u5NtBUW9wH7tlAs02wtdtCe7occSQwfPWz+v1FY7PU7TpNfuZYVqS298h7IVj4wHbOR37wtKaKy3jn8fWJpBazOvvGKEWq7CT0dYTm1G+PUBqKcg41SyRYUQDaS6/ihtcIcEURI3lpAncC7YklcleRDfCBJzMO6VzcvHT/Nby4YROejvWNDeOYv7i2FA4+DRHetMOuA7VQTiOaOYBu7AJP+gEcBoHjAA2uh58PWqzj2qT3bUu4O+hpBFpgiNhqyeCCs4khGYeZuEHm2k2Yb2LIwGtm6ll9m3Ci+mg9Xx1aq5gZ/lkw/Lv03fvKfv5/DRbups3ZBhWGSnszKZ2E0ZzYfOSe9kMQ80Lf+v9J56PLkXvjVVROGu1hkpoj9n8y3yiA26U2kI1ne8oY63uqPf4cz32/KIfJtdB3VVXgfnjOaD15IE3SXTL8bnIwx/skCvQGk7e0ZsgUegN6FGNelP7XnYrvRh/m7NnPzJyZwcEk8nKWP7ne6oexqwxVT5WZPdG50Qoh7tuh+tkSHWkM6lcquyd83QuMBoDdpxQR5E1vPcTVN3WjUT+6HHZ3wjh7bvcDTV1OOY+aRy2vdPqPv+0g7+d8XYTvw1hseiJuWNN09DWi1Osb0RZxqVKnqFalKjZ4/U+MlFWZ9HRnj5tROR4dpGXeb8sCDAOV0e4Whn/C/hAJq0xYA+2Tl6mWtb4f4bmykt3g==\", \"config/data.json\": \"eNqNVMluGzEMvQfIPxiDHpwgjR1n6XLJoUGBAj0ESG+JK9ASPRaskSYSx44b999LaZaMDQeo4QNHfKQovke+Hh8NBpkCgoCUfR28xu/mRCjt+SgbLSHPDY60LSsaSS2VcmEyvvjysQT/XHHcWRM11wZFCUTobYw8PR2dnu+DCHyOJKQzVRFRtjLmoE9IsEpzJRgY9pj9hBma7GyQmdb4ZiCEaMjWqBNE61dtTdvUKUjMvStELNNCgULPRaFD0DbnC8hX2IIDFCU/Rat3Sun88aaH9DH4cRc/vhu37plsdRXk3lXl4YQ1YsCZ2/g2Kp49uMpLvuA+Zr3DQNoCaWebk8Z97zztA9qze+/I8c3ZtM463X1oECX61JbDfJAjMMK7ddjzp3BunwiIin2T8eQmuv4mQBbY2VcVedCR8vH5p3GbYgUm9oGrTY6L6+5ufsfeUe+W7ohzEuabKDeoyIm6y7AGj53kPD5X2qMAY0SSCr8YQS5EW2Dkvld36bH0TmIrjbZ+fGG81CSUdy2Tib+un8mRtNWMwQ692fD39lE8PYXpyZx5Htb2rVYntx+yswOoYfByGxK/Jy16qMttybyevBukAm3Vmwr+P5J0wWHM6WF30IU2BvyCiBF7SrJVgV5L4SriNSEUbcqopowfCnQ56biwYIW2c1E6bmWijZttQHLHXNCkV0yTVcJiDumDsdpq2oi1poWI0UvEMhlz54XR+YLyWdHlDxJMzVpmncWsR2tdWo/PqOgkfd5R8XxyM7m4umozSVewEFgDSZrZn0Cqn63AwvlNLxtLJi3AZjl0yZu5uh7zr9tGTdkMgKUoKkOapYVx316ed6gCXgSsQPPuigmBV5cH2c3K53GvHKjUzqzNQC7RxnHJFLISC+4iS0IKLtPN39r1zBOAYgbE49BM+E6lc75eOF5Y3oVQD4xwK/QGyr2l2SKbPfcuPhV9fMT/f0dF9KY=\", \"config/data.smoke.json\": \"eNqNVE1vEzEQvSP1P0QrDklVmjSlpXDpAYSExA1ubbCm9mQzitd2bW/S0PDfGXuzZlu1EtIexp4347dvPh6P3oxGlYIIAWP1afSYzocbocjzVTVdQ11rnJJxbZxKkkrZMJ+dfXznwN+3HHfSBS1Jo3AQI3qTAo+Pp8enBXPIHMHXGIW0um0SyrRav+gTEowiJoKBYTfVd7hDXZ2MKt0bnzWEkAzZG12CZP3srEWfOgeJpbeNSDQNNChoKRoKgUzND0TfYg8O0Dj+FVKvUCn+9NKPfBh9+5IOX7XdDky2CoPa29a9nPCmxC0WTzkE4dBnxoybz2az5P2TMVVwmoY1ix4oKTo7/TDrs2xAp2fIdo6ziyI1hvjsKiCq/Mr8slxxzoj1LlUT2mhF9xOwBY+loh7vW/IoQGuRK8GsEeRK9ASTtAPezqPzVmKvfM8fHxgvKQrlbS9Ulqdokh25dIcuy+7Oye7xr/2NuL0Ni8mS5Rx39jWpyfXbnuwT1Dh4uQ+29RInPXpMbu+sj5NXg1SIe8Xqkcm6/n9kpIbDuKwvuwM1pDX4VYyM6ADlz03boCcpbBt5CIWKO5c6ouIfhXg+L7UwYASZpXCWpcxlY7E1SFbMBoq04TIZJQzWkA+MJUNxJ7YUVyJFrxFdNpbWC031KtZ3TckfJDdUrlplrMFqUNaO2qCe3m679uUVkO7PLs+v3veJpG24D7gFcmdWv0NUw2QNNtbvBsm4Y/J6OYxeyT0YjTLqB9Lsh7VoWh2JGwvTLjs/LagGHgRsgHgxpHzAe8GDLJNyNZw0aNWTSbsDuUaThqUK99zj/2ahO4o7iNz/iSRjLgbclvygsDz/3obQDYiwG/Qa3LMd1CMPa+NVfKZ59Ia/v9Ulv/I=\", \"config/orchestration.json\": \"eNp1kU1LAzEQhu+F/oey5y4mqUrr0UsvFTx5DdPNmI3dTNZkUhTxv5vdbqUWPAQS5nk/mHzNZ4tFdQBrO9QHjIRd9bCoTCZLNn8iqbUUcrORN10Gqo/ldM62bPe+7iG+Z+TaOm7zvj6qajm6GWBIyDqFHBv8x65xjTEhqfI6G03yPoY3bFgT+FG8G4JfytkNwdvHp/p5Ct5eBcdM2plBc+6oX3PXaSXUvVgrMWEM0ZZ6jjECu0BFIIU4zVLTosllF95RZkxltppGERsk1n1OrbYZorlgbu9+G5Ajq1uEyHuEsgSG4taWVQycXJ04Dx/OZ68TplQaaGBG3/OISHHFMFiCkhyxXOMIXdlE7EP8A0i1vPzZcw47jyGXVtgEMmPzlRJiPvuez34AnqCpCw==\", \"config/report.json\": \"eNpdj80OgjAQhO8kvAPx7AHx5+DLbNayhgbabtptMBre3VJR0R7nm+nMPsqiqjbsqdVKtLOgumh78G4Mm3PVHOv0ttlj8KZNNAkpYA88OIGAhgfKzrr+WunGA2oLKbKyrP4KHfJv0+7DmLyJgnmMJyaUHH5BcQzXJEVPAcTlFYnul+yFrOoM+j5pj1mZNRTVQdB3yjXNYbuAEVMRgxbyue218U0NYUgthqz8W+rZMi2XELX5/OZUFlNZPAHro2Ni\", \"config/train.json\": \"eNqFVdtuG0cMfQ+QfzD0XDWSHAdN3pzWNoy6SZDEaYGiGHB3qd2p5+a5yHaN/nvJmb1lbaCABEE85HB4eMh5fPni6GjlvP0b6ygMaFy9O1pdJTDrb/S9km0XL97/tv4E/jZhXF/I2KVqfditfsiR2jaoxjjF7m2lexDvHXqp0UThrcoeFQRU0qCorYlk7D0DYkPwbrN7UwwNHmSdI2qXeieTtKisDXxaMuy/3Wz6TODVgwjROidNS8geVMCCSV2BAlOj6MA0quArYw325+4RYvIo6GZEgrRmgaeAApQS0YM0lPsukEP0CecUOPCgGXhkI5nzTSmZiA8uF9JWTSwnEmorJlweMqKTirJWEMKIK6rIcLSHyD6bHzcnA8Y8EH5ATne8Hcwa7kWDLnZkXU9WunIDEQT9UtA+szxmAV01INS2ZFiad9n83VEtUxCtCE7J+H0U568kk7c7Ge86cLv3MFC7nUIqaFsu8v9QvOVUYyKMXtZk+bMwJ5RtlWXyeioFem/96q8hoIhpbMSoqJ7L2HmEhsl8PUVE9FSvpBbW825zSdbXrF8l7mTAZ0HSyABOQmQpBoEGKkVaI708xZdjMCPhGWjJ7XM+ufWedG/1c3BHBdqWlCucpYKC/IfvtN39NHWB1X9L+yAS1gjybRYlM4VsLl4sAaZyO+Yo5n540ODdoMP5IQf0lQ0yPnAo2/7t9wDdP2CczZXzSNzRVXJpjfTcUpuiSzG8YtvYW5JNksxPouHtDxIajNxjiIv8g2+APQqN2voHQVtxL9Wyw8pCkxsSeTRbvvDKle1IDNMPbZrV1NBiEBXEuht2x+vN2zdPPFgzLS02J2qoOyShkMrziI8DSINAfnCXi+mvNeOKouobZ6WZ00X/0B9AlZ2ZjxtbG2iHCKl1ilmT2UNwyeFJzTWdcIPoFk67WfqAIZQZfpw2gtSkjs4mnzPvJlnxrhYV0rwQ39KkWLbZYp+US9OE0uRM5xtq6Dzx8Sxnma+lRpPjtom+y0s0Z8KB7pNJE5FUwC8W5ab3KrM3K6FK9Q01Hc2BNfDlWLy//vnXs69j80mpe3k/wz99Pju//GOmz5antsdPf/8iPp9dXH78MDrQblAV1DfiqecvZ+en11dfh4gZG7QM2/LGPY5vyQEVB15+OP84vTD9Wy34jbbNOHcvX9DnP3I8YO0=\", \"config/train.smoke.json\": \"eNqFVcluG0kMvQfIPxg6jxJJToIkt2TGNow4C7IOMAiI6m6qu0a1uRbZGmP+PWT16paBABIE8ZHF4uMj6+7xo5OThfP2XywjGKFx8fpkcZWEWX6n75Wsm3jx9v3yk/DXCePyQsYmFcv9ZvFHjtS2QjXEKXavC92BeOvQS40mgrcqexQioJIGobQmkrHzDIgVwZvV5kVrqHAvyxxRutQ5maShsDbwacmw/3q16jIJrw4QonVOmpqQrVABW0zqQihhSoRGmEq1+MJYg925WxQxeQS6GZEgrZnhKSAIpSB6IQ3lvgnkEH3CKQVOeKEZuGMjmfNNKRnEg8uF1EUV2xMJtQUTLvcZ0UlFWSoRwoArqshwtBeRfVZPVs97jHkgfI+c7nTdm7W4hQpdbMi6HK105UpEAfRLQdvM8pBF6KISoNZthrl5k833jqqZgmghOCXj/SjOX0gmb/N8uGvP7daLntr1GFKIuuYif4fiNacaEmH0siTLPy1zoGytLJPXUQnovfWLn31AK6ahEYOiOi5j41FUTOZ6jIjoqV5JLSyn3eaSrC9ZvwpuZMAHQdJID45CZCkGQCMKRVojvRzj8zGYkPAANOf2IZ/cek+6t/ohuKECbU3KBWepoCD/4zutNy/HLrD6r2kfRMIqIN9qVjJTyObWiyWQqRxytOZueNDgTa/D6SF79IUNMh6yeNn4f7cIqICAcTJYziORR3fJtVXSc09tii7F8JRty6DtDocWk3qSZJoSzXB3HGhh5BZDnHegdw5ii6BRW38A2o5bqeadVlZUuTGRR7Tmiy9cuyWJafqhjbMYG9saoBCxbPod8mz16sWRB2unpgXnoBRlgyQYUjveUycNBPmJm1xNd60JZRRV7pyVZsoa/UO/F6rdnfm4ocWBdglIrVPM2swewCWHo5pLOmGH6GZOm0n6gCG0s3w3bgapSSWNTT5n3ozy4p0NBdLcEN/SpNhutdleaS9Nk0oTNJ6/Xk3Tnk4ytlNWzbubHHcNuiYfwTkTHtHNpgPwy0W56d3K7E0uWKRyR01Hs2cNfDmFt9/+fHf2dWg+CXYrbyf4p89n55d/TwRa8/R2+JsfX+Dz2cXlxw+DA11SFaLcwbHnX2fnb75dfe0jJnzQUqzbt+5ueFP2qDjw8sP5x/Gl6d5s4Lfa5lc1n/L4EX1+AaenYuQ=\", \"data.py\": \"eNrtff1z28iR6O9btf8Dgqq7A7wkLdubTaKE3pJtbeKLLDuW9/alFD4URIISYhJgANC24tP97ddfM5gvkJSzuat69VxlmwBmenp6Zrp7erp74jh+0xSbvCmiVZG/z6+LcZsvi+j5y+fjFy/qi8dHj34Tvcmbv22LLmo3q7Jro2XdRGfl9U33+2evJl9/9fVX724K/haVbZS3bXldFYvoqoCCRbQs8m4L/8/r6kPRtGVdRUb96EXe5S3AnjdQDj4CwLf1RwADVaoCakRX+Sqv5sViFH0ssBb+AghV3azzVfn3YjGJojdNvdjOsX40z6to2xYR/Fd8yufd6OuvFkVXNOuyKtuunEebpt7UDZbNV1GbrzerQuFKnejK6hpAnuXNdRGV1WbbMTZQb160bbH4+qu6KjRRmvpjdN3U202Ud1EedeUa215EHxsAVVRADwY7bjfFvFwiAnnTtdDPOI6ResumXkdZttwimbIsKteIHsCo6o5I0mIp9ba5huptoV9cz/XPm7y9WZVX+vmvbV3ph3Xe3eiHutU/N6u8g76v9Yumh93+DfAunvTPXQNE1o/YVUF/Xq9WBdG/Vfg/r7cVkH0ULYplvl11ixKrUulF3hVEJymqnkcE8u9AXSm4AaShR6rcG+oDfeluNzBM6sNJdTuKXkJr+dUKoLzKN/h1FF0UMEQwdwwCVtv15hYHpdr0NIDxynHmRptF//I2b3Bs8W3uvp1sZPTx69/6r+22K1fY2tdfXbw5e/kuOz95dXoRTaMk7pq8rOJRFH+AWbuggcWnrmi7OP36q1cnF3/87lsoWW0m27Lqvvs2Ofr0g/MHyv3+9Pz07cm70xfZxcmrN2en2Q8v4Z/nr89+fHUOteOMZ3S2LOGfchEHarx9/VOgAvSKy5+eP3/9AkqfnTw7PTMLrvKrYhVz7+YrWOcRsg5eFUDuNzksu7dI8RbWaPIWhh9G8xTI1aTHX38VwR+Y8W/zEpZQlC9htGC5LLY0ZlFbb5t5MUasoyuYOYu8uY0+3sD66YC5/DG/vsZC2BAsceAyVZE3q9uoBgYxkXWE63wZrep8kQGvWZbXCU6fY5y10X/S3Emj8dMI5+ElvBvhrJkJYlwhwwrQVSxLlVP++rGE10aRSb0pqiRuYPxgdtUL6Pw03nbL8a/jFGfEDcynVSGg8U+5tKq32+Wy/DSZA+Nb1qtFkkZToO8E12ts1OoRA5zw4wQ7lzD0tC9XrNrCqdY1t84bQoNn6W2+Xtkfi0/zYtNFL+k7jRh2A94GgDQ4gJE5ukn855NXZ4IqjCex4QYmQtkUMEdu8etvo3+/eH0Ow1YsYPBqgA3rAVgDEHIBNLwFwtG6hjYHCIBYT1A2ZT4VgL7ALGFelFXbobBIuNqIRjs1esHY/0e+2ircn9to1wBnvW07EAkoQuqrvwJfi6Ud6dQC0PkcL1hy4SImBo8/NuaCwBf1tgMBgr/WxbpubvFXvl1A6TsGuS6pLEBsgfawblQbk0W5XBZN0fcm7XsrtXZ1bBm/EtD2wLTCp4+jzwLlTncPS7SAyuUSaNxJs5fSvdllla+LWUryH3+CcIwMLjfT2OXVbfIBMYl+N42OqDw/QgVuI2UJzmJpUrbzVd0WSbtdJ/J9FD2aHI2i/KrNuno1fVSMHz3ePY7EXx/2zPUhclbVJTWim7otu/IDS2hoLurq6FHc01X12B7H2eS66JK4nQNweEyjX8ByrUBMxTsxencDjEorOlcwWaB6wbhg3xFegQKpaEArED2pNbABMaDHQCbS7DIGRt1mm6LJUI+IYTyQyDsR4boTq6JHkn6OAxqVUKLnq3lXr8t5hmwoW4AMBf54iwvxWEnbnqeiyAdNq6JhOO457zlQTPA0CqAoLapusn6/KJuEH9rpu2YLgrz4BCpbVr+nR8GvK5BLoXiYWmCQSWfMWRPzPb+KvgEG2603scnSNShh6B8PZujEjU0yjKQMAmhRj8vbeVlOf8iBN49gIIHJddPHI1rk2fvitjW7hH+4+gS1xiKJ/1IpROt2AnNxlQMX0OhaBE77MVrAQkKJmIl2QkpAmyCjyoC4pigcoW4FIriilzQ8K6D1JX5TQpF4oUhDBcNmuFhiQoPUJv7q/AFaP6+7H1CcK56k1H0ABXwIGGK0qIuWgBEc4EoIVLMk6kHPHElAIz+hHyXz68n1qr5KpD8p4rZhrkL9T1IbZ4J4EK7ntVbyGQ1gVvMb4P6fpa1fNHcRlAdFxkZalhBV6kenrICdg2qy2q6rhP8DJqxUVFw+sHIUry4WarRw1YxwV7NAxla4VWjo+pLSMVQrWEzBJ2ksNRQOAq7eE0H5N5JUULtzmCLh1JNtiZSCBrglYpAMUpU1m0stVYirAm80EN4hxp5rkArJz30rOAQfc55ABDc22lLjgO8VXZqelNTZnq77+6bKHtA1xCfUPQ8jeXNOmx41VUBSwOrIaFspM6YdnjKhqWGVmfXrGx9/9jliUTXD5T1EWdiPb2HXjRqGTVvUJyyyWlqGBXxmUVwgojzPV6uEa/T0t8Bw2TQ8JpcajUCdmTVYlzNjqGjjorZboL3Bfhqkaab3HkR7kOTSam+2ACpYpTWfj//yF1QTHzrsBGBMUEfOrm6BqIns9idXq/x98fgqMewhJMMAjkgw1IKvQWBkLXyd/jqd8GMCH+Kr8jo2JAiQsJ4jnU0jidoh/m1bA/NOGKnNzW1bggKD+8bWUQIAU5iXHRpQOlDfOl0KPnz9lbMLw8L99vBEEGDbDezwPCgsutGaswb9zTLmNLisUOMtkHXwvlCmiQ9mr9Yk+v3ErzqkOhlFpqhdJqjAkeab2kqwRbwJvW5NOXUfNJ+ToUgZstDqtd50t5HanSjUqBOAFhCO0PJ6BbPBeOBaPOLIInoMeGILvAeko1KN6OFDE28lpEcRPXqdBjVnDX1muHdqoq9Bh+fdkDQwJjoyHi6hmrx6T8tIlAMDR7OpUf8eFK/pKl9fLXLpRjJOrJ5YFS+xDPDOf4ksKpFhQICmPQPUGglhdXmsOzMzho87woCjb6bRI/6Ut22B9rVAV9E0wChaRZHXHeHksCDCcz8eB46AzWQYnKFSmqZT21TUJsL0aF2PXI5A7wic/G6LgosSA6g2EzTyNPltv/gvSOz1lltAagO7IjYQldj0GKi7qNdqtiMrgM1NJJxzjfsIc9XjwuiphD9sJHetqwtugmuqFX9VdB+Loor+XjQ1yRy0TimYZAqeo9nT2MNx/emOlpnwQI8cOndd2HMQeHd3uymm2iookJGYIAxZyLPVkEkzuYAvSvonl4ashImBtVJiuWrson+NejOjfNLfnj6NnjxO7SIMcKZmDa3VvnGx9WbwPrFQtGcZVMA1m0CxyfymLuder0lUEe1GanTVZqq9gR3dSh7TdJK3SCFj9WtaoTq9ueWCer2qmX21LVcLntGwSDNookp4b9BrUbQbGtEm41i2TcJXs/nyOrD1HZZtepJ4zHSitYCuTrAp7FKGsuUTKmHElv42kZ0I7lPYNDlZF12O2Eyq7dpk2i5Dkm2IwWd59bAtgCyuU7NbYuxwytiCJLNEnVfZK2QsCK9xU1ujJeW1ENan/SXrgQ6Dw8Oc9RbVOtAvik/z1bY1hHhQWRjAgGfzIUqTGn5eYb7wPYw+tkmZVrRT2rMG7dZtvMaGVBuju59ZdKIiwWs1iMcu+aNEz53FFNTXfolmZHhcl5+++5bVKFibvdwIi5G/w3zksooxuOzAMLyQKgnfQV9sOxjBBK0n07i8BmUaum6QEcEmf4/+L/7z9KlxSPPkKIW+PrCObZ798Mtff/ur7148en767ekvn/0mPQDO41/5cH7z7Yujb3/z7NmjJ08ePXp0+swfizBKjxDUv0Z8pGRsVZCaGW4N2ow2EriFkJ2ETdqR2E0NVkgW4dleKQ6DRSqZOXQMf2g4mI1DN/oOkJiSzm6rktVWgGuAIHRAGkYPI7ZVP37wQAvHZQnDift3+sJduTyaadE5r3l7T8W+sYs9mnmi6uNN0RQJIfI7rjSKjkbeF4Y7ih6Nose9WIJS0Klfm3ssOiAnJcoYEUeZEhXr5xkU6crQ+Etb9xggwTVVKDEGRh/ZciHwlw3spY+jzWKC1r8f8GkUWbaNkE3L68U8r+pKRCiBnADbvTwekWEjseClM9WZGOCRzX4COK+qPIl/d37y1NlZA2Z4bjtBfDM+DM744CfRjbIR95NSQpDfX9cNainETyZdndHRcuLra5qdaWB8lMo87RgVB2XFO9ZiAHAqW8CXt487xZ4655GdDMEmOzYdwBfaJMYfpmTWYbATpM4msQ2kVGxng6e0w9zXnBCXvpuHxqeo25+xs8kJHoXVTb8FeFG278dX+Rw3drQJiOi0LKphThVVAdMNPjx6/OvxVan3vC9ftCTp2QtDuI0cCvNhA9A+gz1Ml2XAW1ZL1uLwQEbMNKzaXaF5t9+9uGcWBMqsd9+TC6GxDSNgPfcb2lago75PDDjYjQnwm4pP83BU2VlDvUwsCMM1AYFivsUzhzdvT37/6iT6aw2jBlN0DXxi+tPJWXyPuu1tNb9p6qrettPz129fHVbb7nn8/O3pybvT6N3Js7PTqMSDE9BEijZK3sMKjN6d/p930flr+Pvj2dlIfb+Nnp29fma8Z3ekl+fvTn9/+tZ4HzuNvXn78tXJ2z9Hfzz9M8HvIYII/enluz+8/vFd9Pb1Ty9fGDW/vE+nr95Ix2haZzTlosTuhYHT4Tj0kzdiI0D/wp5+bumD9EXCdsJTLDMq71IXqSHRgudN3bYs7gC9I6eQ8O6dZQTQYgvfkfm2GSpvZWVWMJZ7vlgwnrLgcWzJHDsyptSx9hm6JJvqbGTIyEEeUGIdYFpIZ/iZ9PCMzn+8QW0aF79DW0LqmI3xHXRGNT1Do/jsEC8O1KszsTRdF4kznmmghm53AvvVolokSVV86hLVkXRknmIY/iAXXb0hCtFBrg93A+zcfisihDvpV7hqgO3brwdZyovTs1NYMj+8ff3KXCxxelD9NfogxC/PL07fvotev41e/h740SkyhNcmNL3y0ug/Ts5+PL2Iku/TWESB0xLNT1lchy17WvoX0I/n76Lnr388f5c8SL0ORScX3JzLm6j2v79+eW4yQSj8vqo/VtHrc/4xwZk9/T46OX8hL1SXpjzkmreEwP/0h1OgCtejqf+7p9/HI7+g4o3Ydb1C0tQpCUpWAU3CcknSXt/WI4UnKv+Pk2/6T6MerCySf+gpxrwwDiywIZ77zZSn7+4aQ8wVauPgOTzC9zrbwdDDCNxjHoSXcj+0jvQWTp4Gh03m1Pdm8e99TvPlI7m7l/N6vS67JDUFFsorsaGIxDpgP3iIsOrn/jRK2IEX9Nb5+yR++qc/xZYtWMy/3JQYcPiBPS0IA1fAG5JWzUtTxJpIeh2maaL6G7JCHNBB2HzDzlH8Zun3QfaG9EASCYV4u+Sc4nFzuwhCHTyMHk3Rbled0ILXT/4RthTH0VVdr4Y9V3t7bobimM/DhjQvS63iRnSlBAW30TC56A2pZxYgdXplYgHzN35RdDDj0V9mAKE7cwcHBF0DBUp85kCD2GvCRDnQgofoHdfYA1zZNR2dfV10N/UiPo5i2ohmov1umnKdN7foxOVyCFkAsExMJDI0Kq5yNE7g0dDx0PAMA9N82WLLQXgDLNyF7VEqjKRXbBhHHiFgbzI8CMKaEEEMArWsMXYqhUH3m/+BarqVFqp+DggE8h3NSrRfiHts1juSZmWb0Zn6ni4NQUI/1HvBMJv+UkB3xvOdyWfE0xZGN8RNPVFFpVPTfPPCPBEeMuPQC3TOqJfEM3P7IFnL3XFXj5mlLLfVXOJ/GAoFFaHKKDxCK2IArrf0UMTR6mN+i955sLlZRFe3dCjLVYtiUYijFZVWraAjJuze0KZI+5WiD1vq6qj7WEfK+1pFPU0idOeVM/gPNUjCKGeeMr4qVyuAOcbj34s/nZXkWbYoPskmcAOsvWg+kIXuRhxbxKKJC0v6f73F49+uKCaaiP8T+oHjfvZzS+cB8D+nsMM99qBQsz8OMHrNVch0NygFHFcImlpk4s7UrMpown+hVDj6Yv5/dH/ufvS/zMh96Xef3uzrxYAQHAZOky/m2ZeYOtD/lySWJCGg85ti/v4f7n9grR3a7aGqh/Y2UP8LxWXQeTejCEFsUEI7YH9VrIBRb/LJD/iLIOFUs4/n4DNuWFp0XEf8r4uGa9JrUsfNInT2B0JlVxlspCC/Fl2kx7N3GM1aGNN1nlEx4Pj0RPhe0E+HIat+GG6br/JNRGdJm95LGz1oWxSnDJa0tKb4K3l4ra/K6229bVFEwvDAbqugGNIS4/va3oeLqx6HGkdHGsP/mEqini/YG+5Apsctnn0xNdjj2DoAMyyYfaW9tnEdHU0No7d8B1O47V0xDXpI3KPRa9UQB4S6iGjMyY+HaLHXX97BByRUpBmiCdJAC9Zy/+EXzZ2JBzd72X+f0dE5vHSCHbCYsQoWTb3JmiJvYUATFWUZ9mDnk0Q74GHX+fAo8qaorF8rbisUmiTux2x51x7xI2OChZ7VTC8+IR3puOGzcYpqBHLabr4mNuwZpSAweaR78Si6nClvGIktoTC8pkBj0QZ9vggoaH2r/Lqdwnu2gj0/uTjd3yi1heOcKeDc4sxYZBmoz2hF2c0UZF4whyM2g6TgKdQvLeZFyot+1K9N3ZLjAYQIWqsciW4ucImNO+6DGIzjkmBoght9S5+1L6172ortX3IhhCwObFItNkNtNTBz1MtKT419kFU55Xq24C2LFaMZbhBbsVfFnqao8EOQEx9l8wTkb3C9AnuCVusKcEDaWYkfdAoEFwk6MlJRVhP5Xzl50W+cfGrychAWFW2LvJnfqECSdEQLPN0RJKPgTbnNf7zJ3XRaxj11HmJQc9uBOvRQqMJsWwfLqdCvO4880JcByY8TXrWXHu9DpoKdpIoJVaycfEqibdVuNxvydccpowJKASmvGVqCNo6etV4tKnUoKNSyubqUGTGiI3Pl97we+A3tRsWrMDFC93Y70wp4VuhlJ8srIcy79f5PeeyGAk+8CBNjA9kU87pZKD5iFzFZiuXeemS+Q37cFLK9PPLd/93wQpUoYhqFnHgHQhbkmEzq7vT0Vc7D2MA+Z2LLzkOGU7M1i7CXCo55DoVO2Ro1zGPQ5V2COhRFFhkFDfJ9M7UbcwsZ9MSiAMfsG42WmqD+vr27AXVeYeoZC02KQrmhoBDeOZo4svmzf/Z2QCK2j4dHSTmiuVVx9bBhxaktQysfU79JRSYO/0IUgVbmtkXoL7wDuAGotp+4tAyaMSwP7LXHjqfEZpKBGPjLWEHmcPKMSsezlGV5PwHI4SsDbR+tCYPNJ7o99pB88rgHFH0TPTYx6p3p+rakMYyDXqNTWLbhNEZ9k0FCfBNAT/Qu4PeZHBKgnFTuo4ockr8B6LDCkP7rq3XmVIlnHlK0m6U1BUWFDIMoP3BxEHDcMi5wSi8z+VA26Lye8Xu1rNEL/SNxJWyEv03yD3m5oviZB4PdWeefMl0ua/J1tmxy2jlDh2xxYKzAWDF9SeoEE/KRMRtjMy5S2WtWhYRYmNM75mnhLsB+sphlrUkLpaznQPOBZeOyHbNWPzShyQOVQ6/DAPTYsslKARga/DAQPdOseRQAZn03YQ3O1WN3spm1cBIwnVRTxpyiD6lbvJ9BVCXLu0xmiF1bl7MgyOTt+6HalQ/W0GIOmK7GMYCxoM1KBhJYE2uYMOjrFgCoUmAhTeTnRP1ILDTnm61ViNkj7LrZ8qLer/P5TYluHNbUxYlP4oqE2khH5PRHKxc4VUBH6H7CLBDNccB91T6qMeAbalX/Us1yysOBio5Z3kgGwqqXoS2iUypRDVOVOUqWVug8DQ1UKN6+AxOiXbxxTL/hxExAfvTuJafnYOHBUynJRoH/eedVupdQwHjyIJj5T6Y2CVx3yi0e/1h7UlIZTSd20hiN7GYJlkiDgJSeJUnRPE9eobPTmKOf7mkOtVA+pJUAGFRHqaA9nDhVP9+lagt+HHBWoXKXBIy0YoISCmTmhmaB3iAHvQIKGp3+TABZDHjNpD3+Tl6hO5dYzlyCFpw31kkW64581CQIEEkDUQlBB5qlBBuQLS+YOiA8daRfSnklIDsmh6I2KMEkJp3i2nHXr/F0GpjeoVFdrrbtDfstWa4n8t6h0AAxKNbc7+YewsCivCKjwxQJDjrIPBcvQBsMbOEo9irjYAvHeZ7PU5E8Cl56eO8zTq0jK0QBmJQURBKAMZugu1DiWg+MntgwAizmeDbBs9+Oe5PgFtrtUoAGuOAU6BmS3O4w7uQtT+GheTG1KxpnpPOJ5Gu0fdAUhb50qfCKV/5HJhMQlHawCWP6PwpubikLUSy+Oxj2xaziYbSMEc74MwM6PvpucaeyMxr2DyNhUjQ1BMpD3Uaw7JeEePw8CaqIoxPjAVVH+mNmgLKjkXqZN3VFogFP7VB9Y4QG3IcVe6GmvhUC86D1zCp4MmHlCezPJmDTgKlQeTyWoBHCgkL+/1kjYp1BHJIEyzC4kH3eNQ8YdpAR7cNbtSMh7DEXiuic1uBYJo67kNh2GD2+CzBuV2gNmz9Dxc0JMwp1yWfoGaaIGDq4HJK1x57T8X5e//Px9wP4dci11mefs8FSDoc8uqdrVE/ZdDAP6kUHHTQi2iTDKUiFMe9K7SOaiCKPoy395qjDcQuiB4h58ST68e3Z3lg2N9keTJBtRa5BfJbyJBu0qQbnBiX0xUP+TQ3rXmUNvXhCHVOIaJLYnNV4crVvwgnVbvrhhvc8gS/SRKIxxmx6SLwFObA05aJQU6anxvviVgihlkWvvvQRlWYYLLdHCKnKgh36lFqD8/CzXmsrPir+t4f/lt7Frusuo61mjq2325nZ+JSyvkburF11AZkCVjtmFWR0EK9YFePEsKl/SK0KhBNasVeCt+JlE0AnhQKfDgnpdTyirYHKPuOeGJCduyGmYcAKJYntBTceC2KxS2bIs6H1DkRY1B8rYnuUt08TQgFLR/6M6+X3SGdnDcRc2l3FFbdFLQKDKVBUrorOC6cwk4CFAhIoI6KYOHiEQISwcXHNy0y/tY9k1VtxulnnVbmEHvF7J5Dg+N7EqliYBQjFZ7FDRFJOKEIoc36LLMJBlJVmRs36qy7EOAFhAcOJRBPODOGPcRpoGRoul/ncb/2Lm6PjCyKV1V6bf6AsHkQC3dpQntNQ65JBOkB+Zzkbvu1uZlXabUuzKt/UPfsX4B1GL2FHggbeD1rXp/Ug4kKtB55YKpETWwP7N8G+1+R2ZWe4YQMhcVflVS681j2B4UV5LNh4ByDbBpXvrFTxiCFvPmuhZU5PxA7pvE13wzD6LfWNN17d7WZBVsccbdMqq/2kqj8mKrH9ZNvN00nZ1mgoRNXSdjw7YBrxyGVAxL0TSQbk/lMIvgENUBUhCem1mZpZDXBExM5/s63et4nJHSyzPhFQxCFNIB1+y/44doYIPMzV2ZMOOEWtl8u2MEKHzWxygSNIFd3EvuQ6nHbgPM42gvSg3eRF/ZdAXLUT/kobDzq35SZJB9BtJvpXyltATEvheKoN49FvbKKnfTFXxAkCvM9kS0iftk7sH+52VE42Uc0284aKI53jjGNVmQWsavJpoj3UFrDbwtjddCeukvnjvwaBvC+KzTSmZC1xajR8S65IS55hPGOMbSbPoGETXJDenvalyoxtOL23BB38ON4S+9we1KGrk8Y54PaWkTZ6HNiUBA3uC5h16EaSrWvoEeYnOeajQq/0sFuFvQcgU02Pb+qn/D7UnGIkFiN7Gx9dqtSWMqziE24VkYz1dg7pHemxpRGGjUjHMyvV2yUd3yj3Oa3EuhmyA2C05GUHOlo3wXRu4m7Hl454HiUzN3us+OUhuH0+e6z7szeodpBVEHntYF5dXtJFx05LkkBSWoFqXtMawBpmTmawBdKaTfXZq3p5RIbNPmGvOknoTxfa3pePkFOLWCcyddJpm0iMBjPSiUNf6gyu9TXrMxfrgdZJ4VTaamKyyAG8lngLQPmVcOTI7bJcZnLXAij+HIC6M8HOeR1ZHoiSVxrbVDDHi6IBmbxgfwZOagcTnHbLynRmeQtqKeilld5DOrO4SRtyITXy1qllqPZYmKvn+lZ2WWachabdkDejTxCjutitMOm3vo7jaose20aicokQU13/WOBFVGZ6cBPg1Iz7UI47fEPEYJ9gcn62OkX3eoCcNt853qvsRKd8okeOF23IYdpfX8pZ2nGQVh7RuLJGQ7dYeClrezktGXQxJeC2Ag6dPDCXnn0ViYlOas5jswOmHyqbKooWg+nQUxZY8K3kzY5sZ3U7c6v2+t2zVJwbNPSosyQWl3vJ84iROz1LNj3ecBoMpQRlr0VrbaRK8taSQdN1grQqkh4g3UnVAI1sBGyIl8qFhVyggBNBE5bzZVjj58qWURA3Db6xQhNhCe0XzaYBdTuTTae9g4sZW9jRCNoBp4ZLWwcKRQaJOX6vk2Ioskes9fdIhRqCYvqyhPwX3ZigkPHLS6lK2XADLkpt751kOffw8o1VBm3zk7WidQCdWuFWBldj0GCwVCr49iZ//MvvksA1JZhe0Btm9zYS3K2BaoQpgdppEo+Qpx3DXs/NJy98ZHJTfFKZ5PXUVXbNgEY6UXZSvqLDL2BtYuiYszdhGjY84sy2Qc/oHN/OY7zYyT3eFuu6cw3yhu1bX05CV39Q/K4YIhQveGh77qeHoLvX/ijGOH3nV5skzoIOGg9T3kJ26DXv3WOjF7x7y5MWcofd78T6Bsa/ok9RoaMxkOfZ3TRKirJg0YVGGGullh6PziJOynmOL1FuJLoV5cjCNDVqixUb37PRE54Gu2VnRmaeW1YfYIjr5nbQUZxyTg5g4sCwFCXH+iR0CwNy7VcmHDocQ1Ob4zOWBI+xDS7TX9Ji3illnBeDzjuys4bvv4Jq0HVsGu6ZcdYw7CM2DfEQwxZ9Ly5i5+tXWX7m74tKRQWqXvLtbHLllxQhcz2l/4gNjdeBMnVDxDka3Lz9RNIkkPPbdFcmBbUmKLQkcMwTSFfPiQdU1Ao1RcyMMxxgxP26+C3fjErJ/ATvaQhn3boVNCLoA+aBFJ7OEYnDriaCV2YmEpTcOg7hL2MvzaB1aJT+b+cw10obpgAQd+tw06pQPOp3EtA9fGPunPZMYuvg0F8RhjU/ZtMSbzPF+ce2dRsu2IoWVo7BXZEsXx5toq1nUpl29vYpQ8iloIIZtHWyjqmrCaZDF+84zQbDau6bTT8UMYPpz507CwbyoO/Lee9oOnaei4W6bEPMcB5RD81VPwx25xUf4WSSQnTnzo/RzjCikb1wfMAOwVnGqDuX7YygOujbtOPitAofRjjHEKFT1Z4c/UUcDFaD/8byFgpfy+E6zzjE3j00pF0Vy06mKV6SQXGManNuAxsZ+bgZQyQxukjECCRO03ALDbrJf2kTFhEOa0+TvqzkrGXq0uUSARwTYrMwEJWamj2HbXhDGU9ZDK3UJJG2hsuGGU5/uIkZbuuKk8R5fRr7o+GkgHcS0A0sLP9ERjfsH8iEJ7CHXHBq2ubModnItkVy6rooGkyzeCk752I9ix4EVoROuO5DHMgYaTVEwC4ZudlknW8SN2t66qV290EaO48Jn8km/EqlCMQjNVLh0/DCNXMShTHmhEiAsZ/q3rHNDQwY5eHH4dp5K0OfYd/IOelrEQeSWSONd16gP3tydAgnM9HdfXuBXmZfgLcy400tR+OEPQVlQeDvNJwSWVn/yipkODT/yPkCtcNp+zlWm6ae1J2N+BK5dgp7owIUjTiU4l9uofAvFgk3eflfyAraJbrvSUaHNp3xcFR5Fa4q6GnEvPtN7huvOUj5y113ywuay+1qlZirXg/8/mk02Ex/If2Mw3LCjFrVl5voKaMF2UVlbevxGZghOFNHvdtrQfCQNxjWgCFhYrJ/Gvi8A66E14AlsjKmkeT03CO/NKRdcmu3LJKN2MRMCucuQENujPZgtp/ZuY1KajhmJ4e2pOWSZvYybq6Yk3mN7mQDkAxbj/K+FjbvNLKLzfdmFOW6Lb7GMs88vMKxHGlQmTRV1wF/goCOwkgbTFQJAZpjGjMfhh3SYdHKsUYNBNMfEFDvhfWWi+ym+IR27vizykZ49Oi7T3ehzMn3CsdXYZlVVSyMHYky3g9uMQaODvqxwN5ZL1ybf2DXag/mL3YnSwhtLfxAiAHlj+8cJKsNBeWvy5btzhQaoRq5ozuyTaTuEKvPAyjdxXu3WrIKLOd2x7vf3KwjCxjcb4utKnyU1B+SkPfbQOy47xJoGjZC0Azz/7Fp+w8V9r0CZfOzxzGQw6kNk7auaLwMVrJM08G0eZqpcQhoorZjAT6nbu0cjpf0wd/tWMm9mfrY4xXBVUmmW+iF4Ctzx4oYHYzkDAH8x3wmAwd197Ko7bSsWd7A7sxODwTyj5nnetCUqD1+I1kD6FRpQaElt8aBFXyQg/XPoQbuHn7uWyA+onhK7DM+3zHMv6oRh0p/TigaM+BPNsQTLQ+1N/m2Ld5ibHlrX+Zrs0irTrTBStDnfFkAHfik/4CeKyIxB9jJH2V+60TIvaGGDe1TrRhJUlszfag+rMWrJn1Lupleuc+cItEDyDn5Ki7JJoyrin6JnmDrsAFepLKreV4CVjuer0AghsHwGdkTNWaGiARYg5/p68YOGgIEzGCRsA8o7NvooFOFRTruzbro5en589cvYJ9xdvLs9MzYYvBuclNvEmsvkZLBweq/Y8R58thpxwz2JMuIGeXJppLB8M4vDvHcd35nNxEOngzQjAnPXhIz1R/bR+KfxGnNCBMx2LpRQ24SN6U0a18ekpWIIx0ei3QiVW27DkpSfcX2jtQD3vXcRjP2Hd103TvQ99LVOFkc4id2tLJFbOrd6k0JkJ5GR4wUPuhoK6dZjFbjq/gY94XchqCEM5dX+p84IBrE8VmG6egVINl+Svl+gOKjh2plRsGSRZsV+fxGUjKPKBuvqTYLKXAGoR+4oG33+jkD0n2ikEUUSoDdulY+gi0QQkrc6QBK29Vj6qQ0MqfmQGKjJRJW0klztuLY9+9RjqYhNx/mLzwREIYUNYyydCIUm17KxJ7CiZDIxoi23rpZUF6fkNeRKi7OZJgykN8cR/c0Hw3Z2u6sPDlyOyZu7wx/qEvLUQq30BsQ0L3D0+XnWNONf2CkNjkU0rYUf9wZnosjeWd4MJo+iFpXtpqtckwPvcxuQIHB7VK8iwRSdlOvyvltbMFpoXNcHVM2hkeHrRE8ifxS5foqh60asAMDF7+YJY8yyagUjDS0XFkCvoFG7zypH4hRVOnHxMfIWSzq9dA6UUfieCwcc3ZYOiF23Qe1U/3xLod7OwGUbHTxikqo5l626Hk09Jdy800udFga3+8o1b/8KkpitIbGQwe8/eFu3K7r97iVKEAT8LwRSOH0+me6LRx78AMcx3BUOPa7EqQf2eKPHYeGfYTwfGqcMVH3Jbij8qZoxkQavhGdPFFW2FDbjdlTF9hXZI6WulgdZ+jVrTYYoUWGDSS/9a7z1Pe7ywkj3tGEu0bYHKBiBqwsEl1ojT58HHSfY8ghhugjeuN2U8wxN4VcHqKunQVt/Z4zJnSwb2cTQykGzIq5wNDyNyxu7e49unljREGHDLws3WhMjsFAfrvL4D+KdtnpRx5Mahh78Ozs5I+nj6/G332Lt/YaCcdleY+1HwipdqEMmQwIretjzHK7sMde8swA06eLXmxHclowZlClRSCR258Pu+TDuGcmdgzlsrDt4kz/8csXfNkMhQtgWmFK18t7NhH7u6+hMJ4GLsPY6SNsJDWl9R0+gfPoTud1SBx2t7yfc6h3SwTpnjhfe03UC+ml+d/bxX42C9hdsCVgn6IN8jmrmgzy0osDLhrkHJigsVhiypJe5ap6kRe6MsarqZMwagEHGko3DKDChnLRl7NFiQfdV1u5YYYuEwpdV2PNdMvfjhLd0LOV9vBnsOQN6RpKOfDUjLDf8he4wpmeart9DwOJGnaWH9asBiqE0jvsrBAkglHFPUXYsWXWqRHcdNJ77ZmeVNhnkjfM8dpn3RcDh1niv9gKf28LvGd9/+dxmftb1H9Wa/o/mn0g/af6mf6TjcpfYlB2bMm6E8qu/ttIZY9FHEGlKpqGL4jLoyWsoZvoj/n1Nd4oKZa34C2wEsuhFrsRB543LTDo5hqmGUZWwy96NTnHi3Pwcpg++UFLvv66xElzvUUF9g19wdiQeVNukL1Ms2xRz7MsNavSmX0udZJ4PJaIrpFKJzqVGK+H5AFr8uQBAFhujPHNPQhK6rCzEhsSnGpixGyp6T2tyi5ovJGtBIAhpxPKUHI4Hqy1j0lrH9OG6QvgGIJ0PMZ94FiFpRtritNoo08cqgEdyGvz602x2kxhEtIMlLBIS9HF1aASX+FmyAxwVRuuaFWuMSWiHeAwRMAn/tDvp1ezrcbl4l5V1vmncr1dj2HH1WjykkbZQzmaHO3Btqs3Y1ahxqBjbznoOATqSQ9LX6ZFIM011q883Gzymuvjw7AM5+IwKlhuK2vlsLddJwmWmOxy6x4RxMnOHaKUwdmjVEMzoNpu+Gn0aGcAiL88RlFgro9oVtlTlnbM6y0mlVfX0rTA5+PUTE2B/joUnkNPTAD+bUTE4EuV+2BAcfOyOJjZEpjJ9TBsyLvobSdoCRcPpGQ5hI5AmpbuCGVfoA+W5SbQHc9+o7vlftkJxrfrzKwgANXNQyMAhssfSBdvLv0shLENJAfTIdwVmzLmwjr+H8Osr8b7l9CY6QyHeGJBbyRlo5fjzy+u33pVBie0Yvuy9pmh69HDW4VWJWk20J2bojGHEZoGhYJ4MvTEiEilpCc2dvviT/sYRKaLn6HGgGokkBmZ/R2ZKMHIPdEHnkpJDFJcJFJGEgnP18w75ClthppUdskH0ZPvjkC8RGMZChBKal8vQgnKfAclrKETkIdyHF/S6dFZoXLQ3eRV5IrV1Iy6RDM5zEKDBnU7KaoPZQMjRidxb16+OT17eX6aXZxeXLx8fZ69OD15QS9O37x+/gfn7kEPotMNTTMMhOKHEQjWTwlQYiTGIw8GyFxW7/Ef3FyZ0aoacW8D8I20FopYM04n3MRKKsuCM51GPP9GukkV7fhpXmy6XXuIKG+xlBWQR8a1roG+zlM/neKvfqlOHbGcEY7/2Tfbqp7IYUlwJQybUDwzZ6YscKqgzop0Kca52aDBqL/Et69sW5Rml+raZAXljh0Wqm762Lk15og1rxIz11I6nIyCZbMM9bAsUxGyvDAubjG25PRT2SWspgGs/waCE2om\", \"make_report.py\": \"eNq9Pf2P2zayvxfo/6AT0Aep1Xp3m7a4M+ripWnSF1zbBGl79x4MQ9DatFcXWVIleTfbvP3f38zwa0hRXm/Te+lHZJEcDofDmeHMkIrj+I24OpTVJvqh3F0P33/7Y7QXQ1eu+ywS79qqKOviqqzK4S4aiqtKwOui3kRFVUXbcnfoRB9tu2YfFd1Qbov10M/iOP74o48/ord5vj0MUCnPo3LfNt0AjetmKIayqXuspd92u7boemFe7Nbm8V99U5sfVbPblfXO/G568wi4Dtum25sXg9i327KyQIdyb38cDuVGYblu6kG8G6rySmOp3uyLutiJTlVri+Ga1XkNP1XJcNcCUrrgaX2XRc+AQkivLHo5iK4Ymi6LfizalpCnRoeuAmgzGrduCu8UHQye9WHf3kVFH9WtHSpMAbyBf9uNfdkfhrJSwPu3lSi6eqbmUsNPPv4ogj/Fen3oivVd3q+bTmTq5Q3guRN524l12cMEOaVXRVXUa7HJg23XVdH35bZc08zmncDeVNn20qkKM5hXTd+rn/tiGK7FbZ9DlW7diK16b7GAB6BlviUgeX9oGewwrqqF86pZ58Vhbd6lhkXX12L9tm3KetBE+vnJzwPWimDW9uU6RwbMNzANWdRfF59/+VUuuYqa35S/63Y7UeNMCyiuC0BYLg/s6OOPfnj1/ffP30QLzcCznRh+gEfRJfG+eCsUyWLA66laSchBV8X6LTTSzLRcItMBGkO3yqKfmlqsJPiN2MKoiw3hmiCjzok/0+jsG+THuaTCbTlcExvPmlbUiajXzQaQWcSHYXv21zhFjroG3qqEaiCpCSu4pmU4q5pik8gaqe0Z36oBwDTWMPCkOwDJyo5hsSnXwxIQzxCflYK/LloUDxsYo2oRnUexhBHjowN1hjjEsmXfHLq1gHYGRLk1zzPxruyHPkkjUcHiQhySnGYtz9MZTEpT3YgkxbUnYN5DXfK+FAUsgWXnjAJNtxHQb07rIK+LveiTvVztc73s5eCBz1ZEjwowxFeaFNQIxrPEhwgEGb3JIpCcNYy2G8RGg5yVINpgdFn0VtwtqmJ/tSkifDdH8Ak+LS9XabqSkIEwqj2W3hTVQaTUAT0ieA2XXgDgNPrLghBMuqLeiaQCZiH80jTlnFGUQN1/YKPnXdcAKwOXiirX4JBW0cvv+mh/6IfoSpBYhVXRHJTS+F10DbI8IzJ1YwkLyqLc3jGOzmCSB7Frurt5RARdq2Uyj0YL539piRC18WFuyGHa2LHoV9SV7UTNMc1r9F0JkmX4+cmbQ61aIop5XtblkOdJL6ptBiK8JMz8bklcoWxHZtdyPoHaqS0H1GSVWQ9iCdgApiHun8QRckMz6MJaDFWz9l7SsoZ+yzaJz2M+TcGp2sYva5jucgPiDtde9Oubl/PoPeBzHzOMcEyzq8P6rRgAbad/rxKI4m35zlby8LG1h+7OQ07Jz6tmaJ7YEvFuLdohekmFhDSKJ3gbHBrMCap3xYcEKip74KrfDiVKB+T3AhbSk/n5OY13Q5MJUwxij5gRIHOhtwOtAqNp+pmob8oOhB+I7CR++s+f8zfPv3/56idoBzBD5d89f/H01x9+MfU8Sq2rEqXOQg5Y/UxwnjPVL0mQhcIBOiE2Rj7UPLdpbmuSxZLnNqIfQOeg7mUSd8R8O6wD4BYcD0Q7N2WwhGHZ583Vv4A4fX7zOcd+2xxq5N4L/qpDwCRFDJCZehLJt8Q5C8ZFWfSaGGWxjd8zvrkfsyyCRlGmQAtJ32domdVDD7Rarvwm+AdEIg5w6KQcjP8u7uJVOq7XCbAXyxtUIdBkiSKO4ZPOV7MqxL1sqeLiM1AAWf0MDLHpUdWGFqKRNygK64MYl8LSKirAik0qqCUNfKK+UmWz/Vvg60T+6Be/dAeBZjxN6Vv6GRgKZwbNWKQsE2fegEpkeSTUYRoAJPnjs0V06Yg0pBMVBRfuC+jop2Z4gRW0aPqpiRQHKpigVZpbuXjfM6Tuz30e4ovk0NJIpBWnVsqkIgmtGGnHSYby2fW9bg2PJOpwyd7HTMwJFFxFd2cBGHj3s2Hfnr3HLcgM//cF2CLX4h1vTmZpf9gjL1ujk7QTlyfl74JELsnaYgA4MNX49pi85ROuiCSne+hkB1nkzLwzlCx6/m7oiqfdrl+8j38UQ7EphiIG3RFLROFRI39/73GJgeTJoGs0reSUB2UGLOKFg4UHF7gMbRtTZanlxA+i3g3XIABQlRK1YKGaalKkmDFk0fv7VL5TQ6FmejSBlSxZ+OUrpXZ+McNDpSrU1lZaJdG2ABpvDLRzwgb2XGbb5IsZTqB1095pAo3RmCKZYbhs3OYZQPyZ7FiYRgkAZs4BQbJz7k7/PWxjFb2kOQQyaRE/e/X6f2KvE280hMsfmHYzhvCUU/Hx6aYqf/ZUv6DhfOg0E2qVvz7HK3YkpkUlYKP5AStGWVfP6S+0G8Ydyj3r7LboarDmUfUeqo3SevsGdJ5dzYoIKOk+QdXs94z//KfvVEEZDdt/IA3Yob/DFvhQ232jFsnae7IcQFIJtf+dsPRXK2viE5oSGErGblAamXSIo5W1vqVdomrCNokjVSb1rd5hnqjTpM60de9KAbSkt3Ij7++39aZo3wwo4dnWw+DI9vTa1TUzEug7bd4mUmMtwLDbXQ+7q/2ZnKwzud03MzVFEVMhpc3xoY69isrmSPkIEGtjSxwfvKrsKGvuWoBKef8kN/szx7MAxk2NjpHNHIzpplJ8kBmzX76WXpAwz8w1oYvNHZnkTZX4Nv3PT/Jvf3329+e/IMnAGgmUv37z/MXL/461VdRf41rJJXLo3CDwwEAKXdyaYO/SP6FeOrzrQBh7YyzLIG3IeaG6kR6Kx7k8QJ+XtfR4pEvciKw82MtY4RivoBu0I0cV1ARqylNF/UNVRq8avFX+tcQ0ziJmmeL8lhtlwtsBTbiH4C33DSmRLmHgdkpJADTOUtfRoNwm7cgUBNRcblQuAVmCEEGmJbKLDM26sCWYMvfQutlflbUwTNyD2ur6YdJlgfYXDGtz3KdxnKNNX+hTon0U31BJBHQ/KWlTKFo5PGhATPGftbXVCDfJ4wxshlGoM8IFykNOGYaM7twSHPoDXbstxSYv6025Fn1ylw/AZPOobmf1pui6AizZ+rCXLjvRk+ssA330rtwf9upXL1CEwCMhb1taNYPbRgk5jb5emOYjekHbwvrSVAvYuA93rVhAIfTx1ReKQdegNgacNngP46KfCQLoqXPVetQYkC/rigygBRuY5vriFp2lEvS5epiBgZKk0acab1n1t0MzFKr/bdWA9oLGKXSP/SUesrLyMlGQv4ku0ug/okTDWMBvlARqR3h7DQJJtVGdfxMg2hqkbAl2mjBYFEPd1Ogu1JC/iS6ZRlFY2HZLIvgOQCejslV0huTgb9IVvDyG5dd/DMuvFaFPRNXDCvAcI4+o2i12V+9k58Bcm2YPhuG2OFRDDu8T5F6tj8BeXA/kelyqNY5rT7qrYYWId7gEJYNy3uHuo6YvKWw3HqxkSJxsBjDle9mmF+hMA6xm8APWY2LAZWSnL4B3yT2tB2wBrYCzcdGYFmmKCr6tCti6vChAgboONhrorGhbUW+QW9H3nUgUuNMCTOiIISEr2OLdGvZcFQJLjN7ogbJy9CA0URiRj0t36VSTvaZe/ABLmFYA7TZQvA1jI+j2M0HUxOpCa+uoQFzT9GATzynMKF8BWwz5FoBhvGlUQD753hF8Gi0WW/HCFBijyVxFMqmMgCSktD7+yAY2EGET2bAMBVVxktUYZliyb2Am8lZ0eUmmPtAh0fOk9ni6+vrQofHiVMRd2+XFxdGoxGtDYRum1raJcvJILDo02HMAF1upSVaHE56Cl8oGVtHVcRVVEDtQTnbOMbAnt8EVmJNGkBwKdLv8ihQ5o/7Xsqist02i66QzkDzSDpWvnnwuIcpVnaPqxcEpSsDgZMGsbu/U+HCRFWBdshZZxBQWY0OttSy6sJiBnZrbvC3Xbyt3Rd/B4miugihgAUMBf8o8hVKLYwyuYz5AMVC0M9+L/b5omfeEQQfdCQywiG8/i5leBeVXADko7AtvSHWzsaSOAaFQ1jr8+lC/BX661SzvLLZlzBa9rWr81CibadNqpfKFFIJu97Ypl9RgqLbQK0pUCeQzVnEMhslEi5XcCtGqUy8TR8rMStiZLQn8HPtbSVqYlbmAReQAZtPDm8mp0oxi+x/PwhQ0KD/014kv2Q2kYyJd1mTAFD/LoKPDzrTqbbQx9WoyThrXVMJMGqJyZ+NbpxbL8LpxeS0LsZTK1slhOed9sW8rgQwVrIqWgWE29ZdqDugFJlqhuSI/ZOK2WMYSx9ibTI69BaCsSLb+HWCBta5L5JhwA/vbQQyx02w2NLkqSDgkHD2YEFKoZLhRAMboMWFkEf/eDxst51VWiWdzjwaI3VA+TvIo23tf1OUWqAHg39uJjmUWQTyP4mtRbc6aw0Ckj/oWmJF7U2MaOuhmrLsRsMagN1AB5ZqxUiTpg5zfi+4G85FqnMKiklYZMjqOEc0dFzgyw/wIm/DKygqW4mr+CDZkMED9HopKg0BxpNo4taQFKM1ndEzLGYIZoLwEp6riVVmZHNacgUl4Ly9XvMWhh7UHUlbpfa2u55FkFFnz3p29B1kz1zV57oifQ5ToSpkL2BUoLg8flT0OlKm6KOgUzEwLIlniCkVlYop6fb0vOkw9Ms+50gfaGstcMmeRxwemoRE0FpSiJDefzpGz26q526OFZ6oeJaWplXmwXQq5ZZm1zkZkUob60pH8jnB3J8ajvdsRS82aJKI01iPXdo8mLfLpRCqdHql8vOjZrXZXTplKCTSTXAzr61wFEPXuy0yeKYz19kvjqO2F26LbH1pt3uiG8q21A6xNsxdFD+1xev1GrCjUEoxYhizYsBcY1dH9g/3/Jb1wOqC3x3cF3xoux8BEh7EbtSfoo0JtDW+E7JrcvuId1KnusDvs/Ax7P8dunK61Stl2mMe1iFxFOrcjYfvw3Np5aljcniN1ipAQYkAFGTMVlbMf5PetOOb5mjLWUFhQNZNjCQsABEaf72GZK0D0PMBeHaQnuRhXmfrPGOTYBIPTlJc6ey1faOOhFQUYpLKCLJmBgd50dzntTtIZlAUJxGnNqSRxkQbvglJ9Z7Cp3EqlIJz4BBvRSfU/aAI0vU7FbNoAP2HqBOXnnDr4fa/dJImL5dmIROgkhK4uAgNjULD7M3fEgYaaa7xmbAJDvVl+QU+Z/plNc0+Io7PRxkD1Sm5wss0P+0QjiBEvRENzLFhr9VBSQEdFD2TCZBb9No+ICXBXbeslplgLTeXsAO2gJS+KVj3RshT4CxOP6l0ynukZpeyKRCfsSgw//+LTTz93NBg3M4FAQwM6nlI0xvJ5riUpKcagIJ47Au6eG1JMUcyZlHYMufaQD9cYydEWo2HsAhZUL0M2OFZdK4suUtfqJMIg4Hx/hei4ZORVh9yyLo24/fICppIcTIQfvGQz5Am3Ly9A28Xt3748vcnfvkzvRwggi53Wu5GlJ3Zt6o/7JbY9oVfN3qf1aWv7PVboAV2DeWR7k4bwke6AX8NcYoBZZI4DA2wmgQEjNYfddXsY9BZEAqRV7jAqxj643XDuCgQOVIscyYJGHpkF6IxG2WH5jehwt4n7nN3VLNe/89zZgUnLjFXWxzcmGpD+YdVB6kzUhMWHyKqjMTPFuGD+UPKqeb8v1tdlLdxtlaQELNH86g5sfbV4lTq/KTvaxUmxCxKXaqd262S83If9QaYuMk93OBwnLe2HQnTWl2VDczI5xGlnnrWJ/Kc6R9VQ8DxMVUmgYt8Od6HwHndzoMl76KV+hzYYPumTxHHtcP/BRIRw2j9ouv7DrkEFYazoKXrkhCFxtrgzL9VxrOJd2S8uTciQE2BEPM8baMpcj9hmMyvkhoGol0XjgKjrjRw5aAxgeL4Mewt1p0qJOmhmduqcuD7IGEEni9ruA3haeTd4NcbXFFGhiEzEHt19oOZyPOikzY62M4+U6g+zvNaPbaefrO/L2vCuLf/YUKHaPl+Vtcz0HM9V6gcL3ejyX1M/7nhjYkcOcBmd9U5Q2AZqq+i8sTyuYIxSuDQFiRvNo5ukhSLBa6f8BOvgKplnfLiro+5tHwlpWTon2NwRGB/F2s/KZvhLKBOH/E6FZ9hI2+4Gz2DFttP1NCLBatq3p+riLKtXbhzYa4wrVyPKNvpmARzD0SyNIwhK+Iwux0IJtwJVfu9OvjM8f86NNPeXC+qS30C3zpoD7t3Giydj6SR8CY5W0ayD+a4SN5oWZNAAQ7qNYaHtJXp8b4GkBduha/LmpovZJgiMqjox3JJSgpH5KRWqihk7sCQVQeJ64BTbJkyMKYIv1N9+F+iwUUUqhWOy030JAzC9nbjOHMuwk0SYIEDbceTaLowJANGjnx45Su0jA2+7U8eNKDvDfrRgcJ3ko7WSeUszlE8hsyLCCkvxm9W1uXRj4iHkXJ5Ef1ySBDuaObeHL//kVAidA2F1c7uZfVcMxQt0HWkd7ce63NSCU2NeqJ42M8qQDIW+FJVlAA7q6khZ27SJDmil4YCW0UZyn63DK/qEKgVg9B6eFxpNbLfIZe3mpYbTOJ36KpNTmR6hvp3qSxb+qQ77GrVFTudxjQPZZomegMsopdQa7nSge6NiRg5GrBF5MwKEizN3MCzHJURmPIwQ7hJd3eEGhIqeZ0WN4+d1v9WxHG/UmXJ4h7hQD0MeegZW326B1m4YV18Z4TC3WxaHGpyU7qIEilwCTJP5LGkFRhJjbZgB15s2qZGxtnSnQA/4Q4pSXqgSu/nyTt5rVkSpyicniyQCc2p8PwK0pOJc+Tl0IP32WoAAZrh8E8Euj8Z9zlEkXyn6zAL42R+UkabEq6FH0UOHdAWATLvBRHExSHsr2XRNy73ZHF+2qx+hPj2uGTQLUnIJkg92oYSVTiOQ9s2l3JXaqinsWS8NF1Cs/HQ2UKH1aT7gu2yqbNiAfnE+UMWPZQTZTIeq6df9GOJS1QuyBEeMeEJS4dzB2OMKF1v2y+ULjt0j2cPBnqZTUjs4n6yyO6EqOcDqOPXEM3Ox0/nRA47XRZvLw5PGycJimEoATy5/u5W47fNeUOrqhVv0UH4XYRDI7OKZ72EPjkuASU/OIzw6HsTQWejiNqcgaHl10Dm+fvQpfIzZhR3ICmvpWgwJW8nzUVhjDDp84Brs0rqXK3mEsXSNHDttzQeHGYsDniYKV8c/S3cfN7ozY4QBDFb6vcIwA0MKOBGOIKyRGXUcJpZTZQareI/ej89P7M9tDcsek2hoDbldSw5jtrWbLaDeqRU+Hv8knk/IAHHLZCLPxUp7cXivp5NxgNXWt00vEo9/UDTBGvw8PYWeKqlovkK7KzmZBhNoBm7n+LWGNSVz9kHTDqXM4fr5v56+ltmo8+h9AKP70FUFRhp+JvnoqneHjs4p/PfscpWSgiM2Rnl0GRQWRip+pubB4QhQPD45xjBwuxhYws7PcavxHlL/wd12DgOTp+XlaM8pUGywzUYseAVbgbej+0Z+pIAGTcN8dOI4IE4xboBB4qkjw55KEZvD2uZ9hpNB5Z5tOQ/0tgotOBwnohAU99H5OfD0sTRQjywjDRyS8d44VkctA3NcVO9F1QB9IG5t//jxG1nbbH9VvuPQRJ9siCmjYgtaRa4RNovxMS04Mad/WTh5ifwYDEbZehkM8QjDc2Odqk6GrBx7oNaH5rAGcxUfci0Ecxa9nFW7Yw5lLPLKSye3c6XSGzziBpsdhmYPy2CdE0+oS+QKRAmhxKM5Zck5j0uwfGSS5WMTLUn66C2DlkrOroEqPHbToCHl2FzmY9Dv+xFUZXqG9w0WOdo2GKl5zvH2Nw4cYfvD3Ta4+D1248CQl/sGBBLeNpiq7q7hqujBmKh5iDm8KTS5VA5PhpKq/GCm19H2EmMq6mrDRB9uCeCRqRW/8O5T4yIcT7QpB+wilq7kjC5Hg8V7Q97YxYXxmrbAG1OnSkS3PwzmDkasaJ1gR04KPqgbNgIPq1G80ro2yYlKO6cVTtZ7HMs8Wq7M5XVoJjucfG/jiep95vqysMXvZZt4/B/wd3GZ3HTlTt0u4s7r0mnI0vdDiXpenqGin2+yHe1AHXFk05Bo3EaBuWBM/c9iU2P6reWR+xGjmv7/ZPa0Bp5imKUhvgq0yXADX0dnEs1QeC+QDfDQBGiCu3G88duxMclXT1hCj88cjGQ1z1pDoagJARWXbnDIkghbwlKbXjfO2QVQ+WGoUOADzaLNptkCW1CQSMuOb6JLGRW6mF083Ol9GpD1BoFHyvoRjZXIZ+/Dkn/U0FUAMiLkRSfHHsCcuWElmOx4fenCmnPnlpsTZrGyrWTKmouuM4OoxcadYTdGv7nhNXsdqGRMmCs5Yn0VqHPPH5l+nif+HG+qoJspZuv+ht9TRtocJCO8TgL2JrFXLg9SLuJPZl9t47FtpEwiz8M/so7Qdi26siftzLy/JoYTk11lPMErWCkd8ABLYGJOxnEr5XAEtkfjWJeqY24K1pgDx3AcbvSgpWOcrP0SQIlsmTBG+nQwVFPRGkzCYQRwxxXCzu3Gp/IyHpr28gLPN103G+XWxeWVsDq8/5XczTI3wGU6BRQshV7U/UEa+g92+s0ieuKDyqd4NbQUbTPGweylYmEP+sncrDh51PxBpnbOVvkK+4hh+EBcXK5vG/82t0tLQ82/B+nUU/rhI/quC/3YYaJTbmUN3chqbpfnl7LC3r1Tt8+THj/bCLQR8IIi71q1fnRLq2Ov0sXaU5dQOxFcfJFNH9xXgeBs+jD/Hzmcb6OrmuMtMqGY84NhawZMdXFd4v1Ed+523T1fp6o4YW11VbPTzMVtfKWzCYlbE5HsrskrsJ1UBBmbj0b3S7sGp39cwpslc2d7Tz6N4B0Os+GdzphQZjV2uruaqaB3okzHshILvGvKdvYnXz8xiV20aYB0dNNSUw/o09JHzKBCxA5gOIli6nYDvFgt4N5hlzMAm8ME5PJmgy52ssAeAmKuVwgDAcqojmRWNY4Bb5XSedOZSZemPHMFT9b9i75YCLPMRoT7Baoouqnh6svDCfA5AaXggBqILlYAHQypV3KMGze9SaEb8dzpV4jATq2HfXBf7Doh5AV806vkjkR+5mR+H82J15lTIU/vI26SSL0vJGTqswigAy8zduD9gQ8tJH5ab2ZGpDeObFmPtoWupRNO9aJsO5NeGMypPkYThajdYEgRdGzzRhXwnloLCCPzEhA7bO5feZVRqpIkFxr5hrY8o5CoCKWa3PH2EneJl5nMNoRxYSEjBSbjybeWKHrr5Q1q4uS0LVcFzEIyZcbGd4A91j7yWx87TE18GsY4+MEQJrFZ49ltB6Iwx6tDk2Czx7LoUHR45zlN+8JhgU25K0GRfzHFxnjdpPfJDIc4DOtjlNmXddPJ9KhSXe3W7TCCbz21annK7A6dP+wdTwGGxOMpRlt+6Pn6cuuBCF1yyu8GsCZD8Pi9F962dfSR6EAG4+m5kOkMfQuJ2rgsbIwlDSN75CKDMWYn3Gbwh24k+EO3EuhNxhjNY94sutMTOJWdbZWeItzaySd5Qb6xM3RNvD6Z38yo7EbNZnTtg/6ukOtw0d8GslnFzseCEnd5uknUow8MGSATnx46Ck0mpXNhLWGF05sfEBgP+j4DXRs1oLLK2SeJ/n2dSk1DHRqP77+hM5Ol/+H9mZT3h8a3XpuuRp+MOs4JStjmAaW/1IWr6Ra5NQv4wRTT8khnLn0mmujvYZma+sXY9BlT0h0qmkz+iQxmXC294xqrUFvvBMaouVMehKDPFYx7poKVdwwBq+nTE04Tc6oi1IIdlfAbmaJgT+VET2Pc5LbNSEdohHFR9xQuzJAQ/rF28z459RS79+2G8JliI9DRq2Jk9Xwk6J02KKv1LUxz54QoqxUyDPCUtFFmji+ay/8JS4NXOXqDD6+YjSG71tWo+JjSDJ9Xemp1C4e2tJprBUC/VQonmqw+VlYO6/yIfIsfujHax23uayfsVLZ5o9VGqIHSKYGuXlxONAEJhMD/qdZEqCKX5i7oZ8/GUEEOE7bahH2mpKpXzRW6LlinaRD1kRTFLn9odtEPTaAzI0Gdbp7++uzszatnEdHh7NU/3oznfCQugwA0fY7C8MQi614JnImux2Ln9ZszaBtpkei2c+QldqNqM3EYbBCWibqr8lhXpe7qUfLQhWVLEJAvB73Jd0pXHywB2X0g+qYKfcuFEXBL99qQ1XJ8g8ZqGqi8seIUoP5FG6vgxSEPoUmXi5yGowR3HEED7gHs5HUQx3CTF5CcgJkCdQQvC+ooVt4VJObGEAZr+paS1ZH7R45CGl1Rsnrg8hF78Qgb49E7SlbTN48wGLwAV1ZZb0UnMFjm3stjW/CL3aYv6WENeMHKNQSUip2wAXRpwCPF3XJLVW+Vav8UB/tY75TbNmwd8EvxUIkD6oGzpcoRkUXGS+B4ihz3wMj4AHNqOJ7iq237k1y89tJNdInmBP20G17dE+bY0D/8TTdncHy9uzPMWVB9DWIABt1wMAXDvd/ABq+Ae6HFgLearjwrzRuvtV8fTxvvqLeFluMVoMok5qinU/eubg9VRaoq1knDTjP3agl16vqhy1o/4B5WtdhgFDlilhNmczoa5a7Skc1tpyCLwrFE7c10CzMWHfX4XX8JfDHxDWTmoTXrivFkxpmLRUoy67zO2IK13TvRBXYEPQCUT1Y2DspkkbvbfiA4v/Td4I7nN7BRcQXTiLKfKkKxO0MFRgIsJcks+n/5LMAH3/f/+Mg8fQgBUyGPfRJhLJG9C2ida+aPC2g9jZ+qrmEGJvIrWFM2OfSNW8Bth9+YBhro79fPfkIp3hZr7fimtxh4NjWedrsDbrVfU0myEf26K+kTbIs834CMylPeFK8+wo6oTRKfnQFeZ4BXbD9ypRIPrkXVLmCXhN/ucj5wixFY+uyZ/Dbcufwg2Ln6SNLx3uSHkM6G5ow+T1vQpCzMYL5tmkoU9SvCv6ie6rvbZW7xQn239gj8/onuwnw46tG98EQLdXOT7IzPkp27PWwWEv9rRPqz7FdFX66fSSatxI2oFrrk5U8vXmWRsT2Sol/jNiTto09kTfrGFP6SD3N4Al7oix380FRGXPQnkg1i5itu/sfwsMJMLxVMs7FMKSfXfFCKR2jUp8nInbO235CnlQuaiwMNfRlPaq7xd9e8c0waEQKnPpeFN1c+Ua+gqfdFMAuAxTX0qttw3TFagaPPZ3nDz4KD5gnF6jARRc/i702neHLISzSKDjXmp9BnDOVHgYfEYEmfnjGyC/8p8QPgOON5jiZAnOfIX3kez3WCDXLbxx/9H+65kXw=\", \"model.py\": \"eNqtPWmz2zaS31OV/8BhymXSpmg956gZJUqN4zgp79iOy072w6q0LD4RkjimSIXHO/Lm/fftbtwgKL3MrHNYIoFGo7vRFxpQGIZvyt2+//mHt8GPeZ93rA82Td317bDpy6ZOgkt4VpU1C455mx9Yz9ouCeDvttwkQV4XwSavqst886lLwzD8/LPPP9u2zSHIsu3QDy3LsqA8HJu2h7Z10+cItMNW4ummvz2yTn3dbdTHf3ZNrb5UzW5X1jv1/ZD3e/WlZepjd6th9eWBCWw2TVUxmlAn0fmlLVjLih/LTS8aFTD/TZV3HVON1CPdhCFY4z19T2iwP5paDngE/KryUrZ7T+jSG5gvTES+eFHfJsFLoGB+WQGUt/kR3ybBR/b7wOoNMyhVD4fjbZB3QX1Uz47AAHgC/x4LAb/7VLG8rVPOIjWT7UXWbZqWKQZt9mzz6diUdS+bbPK6qUtgZ7bPuz02/PyzN7/8/POrD8FSMiDdsf4NfGRtlGU1yEOWxZ9/9vH9m9e/Zu9evH31EZpGYd/mZR0mQXiVV2VBPMdvPev6EJq/fvfrqw/vXrzJXv7y5re373ifrMsPx4pl2xL+VxbYXj5qm2v5BOjEKoSB/xBngl9xMEDtfT507APSretZEX0YauTJq7Zt2njx+WcB/AEB/ZCXHSuCpq6AlluQ5iAPiqFF8pskaTkcIG1Qs+vgH/luBw1AMjqYixT0zz8r2DbIWpYXGUprhFxfELPjYPY9MlcMfF32e5KJtDmyOgLONgWgvAyHfjv7axgjB/fAy4qJDvinZbCAaloHadXkRcRbxHpoQV6W9YIGGazdbbmL+F8LKU8rWM8JorMmvN6BnIpxym0A0xXtVyHIWHbZNF0PNB/qIoT2f1kGF/O5iRZSMPjvvBo4caPwB6kinO7BYej64JIF7Cbf9EBwAIS8EwNDy0qNDDJb3WZd3xDCMPDJEV9h60C2Dsou2DbtZVkUrMZPQb9nSnEZI8rBCnZVbli4xsmFm+MQnhzsVwMYn1I71CBBwcv3v3mAl4fLvMph7WbEL5oNDQSri4UBYCdbbllOKhJAc+VkNnwYxdVggRyMdLKAHCjIghUNiCHw4zGO8NjAHTSzzQ5YShloJS5YuAC7cywZUwlgBOyKtbcB9AcxI64QwKA7VmUvxyfD0oESkIMfmoJVGX8crnkjXJAl6OuMZAka32lkQpI5FH80JuEiCHeXRR8mRovm8p9IiCt6exiqviTlYbUhvYlQWlhS0G6ezr8236N0Q5sr1sHLLy/MV4f8JivYsd/Dm5n9BsiHViSDv6HzFho8n1uj5ofLIs+qCz6i79VzejUCu0PW9E3GiTnqjThdghqGAb+25iHFbtvmXOoWwYXd9TInZf+gFux3HNp8x1eX4gWuL/s1aF3AvwSWbaDBr+3AzPdll7Ea9TFMLG87hPFTXnVWmz10bnYgH9kRpDbryj+w2cXzv9pYohT/PuRgCP4AyYH2hWc8ZCu+4i2RZMjfi2/MNvyVWAwMLILkpQHsnv91KDtwTsCUdLaQfmK3i+AuZDdHEESGiMiPCYpnx9oresrFHu1sBF3iew0BFRs8SlRHXFP2ukjLnh26KNadYHE7EFHFSAgW5tBUI39qrW+1/lFOIS7eHsUluCqbChZQAbPV4O5dvW8gFcLEQF7AR8uuYaQwjoPlcqoV+gKi1WkLIUwOKLqg2Qb2EFxHWvCUrQLP1zQZaB1NLFBa+j2afFAewRxw/W4ZnDaPpjpLjf5qyDw4Nl2J2glNSsBfA0HBg5GYIO87BmxGnwv5HoX4FX0iuRTld3d5qxekhlqYenPgz0wKwlQ1fDQHMIZQzDA0PehgZfRoawRBVqrDOkE6meAm5MYixZ3qf69JUSMkBv6lMXNh2GE5haS1s2uGMQvOqQN3lYESMJ+BAhlqYRapDfciRQMh7MeWdaxGQ9KB7wv+ohonRQRA7xDxxFRjLRCi4+n18Vqa5Zkyyzp8CnKwzGo4WCYCpFoj0nUH3JgtffwNTGq1NlCS7f+CFCLjloG7XjVo34S1yxhiFt6fFNSXZH9BdgoF0vXgLOi0jEzwsdep1G51uAb3CKkLjit3ETvlYp7ETINInf4eHzM0PGS23XK7n5miN+0fJ+TDiihwQVKNHnMBUaJuJFA9430b3pUGidri4uRU3yrnJFCRuVDyQHCIhFkOE+6vm0CAHLlRiGzk96Wsptzd537QGvohzwxUY+l4URTCu3hCj4JnDbJDXpdbCJlO0Va2eVhgMnJKxVDhmi8GQZVsO4Cj6qIBkk9eg20maC563Yi2HBypCeQikiwkqQwR9ElP3NZ44fu2KXjiJJBCoZlnDbB8jLAff8ufwooDJQA0hg84k+AaAsJuOEJ70NN6EMGTrhnQdmGojAyvwBuKnMkYLVx1gXQ1358UR2NGcgQy9TC5DiAJQAEfSI4B4S1OqmeA3GqtLRh6J2hYJkY/7m87zD8IWcTWq1A+lEGI0Rw0bA0Es1rzZ0bmwOnTNhsI4p1e2CxTr6wOqPElWiARckyYjPVY9nVMoKZECgLP6iICgZeo5hAumHzRjf89iVP8KRrWEZd1BKaQhamCTx48DdA0pP8ElRrpcVeLi7k5eSluFEUSzbrhECHdrhChmJhKH5GrcvxVyCMSUPbomAMDUmoDfqnUP5rREqCf2ZNSo6mmkEM2iO4Pp9/WS0CMV8n9Ug7xQo2zvJOf7hM14PJOfrq3Vyv+83cji8gVu8i0/jAYGR8eXAg1tkCdaFkZdOTGL4WvB5bKtlD8LSXLrHf1Ma2LvG1z2UQ6i+iFQUvUJNhSvOXG43QbGgTULulzcyzg6to0N34c+TvKOAKN+9ZGqwPDf8jNtzrvl71+02xe1wW7Ya2gIaXjINIu+yyLgE3bJGiua3gdhG/yP27f5+3vA+t/EkQLXZsjZH2bUicQTvobR9TAQbuiPCr4JSJAHCFoxyJF5v6Evl7wL/z6kbWlrWO5QdUDpejuRwQoCfIOHfcDW2JkaeU5PTPQaU016KwqP7EEJXiWFwW06Si1eVWy66ABVaDtjICFHOhV7n5ERZsyieqeFWXLU50Jh7BQOevVyLiD6XeESLVFQTIi7Wboj0OfnWoNZEWeAXvwL9HXx0ph6jh2Z0ITUAOCLKS7LBNnESqMHVkxCRIsLfq4LbE/WkRy0PAb12/4SYRbfbd2OlmUkLbeeugi5COh7Oh9ByhMAQQSVqyOJuGSl6Ra2K/Okvx1DfAohuCgJRYzikFJ/3agM8CDb0dUz2DQXb/HiYFOyzvSadFKROs9N+nSgEgCay6AQBaYn1pCZ+jzzVcj+M12C6pWwAdp2ICvW8N/UbSar0mRboYDGi8LnTh2AZVVswEgprKiLrG92qC/UCkkyaUdYHKNgZOzkFvNLtYKzt/BBzmytr/VYLt9fmQaaD+AkUcScd08HkAyMk6meGohTfseSE6hCvFjJvShDJxMfWgMSCoOV0uRSii4xBzjPF5fz9AIRJqPKz3mWnlUCW73DYe6W45nMPJtpJTTskKsUtGZRHsMwCvVL8D1a9FEC8kW+hlWVgEaV7lj5EmQhuDuIu6n4e6OKd2CFRwViPRASCMQtm3V5P2Xzy0GkNlwrZC2IAsKnoDIPA6yN6SIvOCu5KizjMyOsEIRSQlfHJiMiW1q8Z4OKXgOC3dYx2uSwManVh2r0AMeIdJV5Ya5qgSatIAebv/g/9mRnHloD1BBt4KjqSQ5Po1jm9c7WCEuvJN4duxhExczcCCBvPNlObs4jdv1nrUs0g+/C+aJ0eapuVrVY5tNiEx9G9lAYrCfxpPvtf6OY7/OJrUlzaSwhXxHB57j3huob8yxEjHDeGSAEb7G0BmCHY79LdcESlFE1vr12R2DrHplWJjzZcShAx5yURD/+OO0bzLaU488sMwF2Q1Vz3lC/aLIntCksoxPYik1F8orh96xvN3sRTrSVvQXi7XBZRDSsmDLsKWEZmzvEGiNiCYPwA51CSyLzPFcJggSOxK4rfK+buo/WNtENrZLYxSH8GDuIHwzIanPK3eYdTCzra2pzp04TRgM3lxbHmnwBS4jKUDeecZdCgWL1nnloLyeFgy0LMfbJVeo5zWiIXfOGzOxJkOESGA7Xwt9Pi3+fmFX/S3MJjDQo+tlxwE45vPM+NbYOj1YoWjuLg9Zl2+Z3Tk65dyTmVIRpo5vXgLLe4aan+pkuEwHLz6+fP064F5svtmwI+YBLm9V4vRxF/zXx1/egdS0JYTRIMk8ziEjAoi56SmusRtYV2WdV7h+GAgCw/3gaNK57aphB4BalnbDZdSGq/99MfuffPbHfPa3bP2UqlbChPwWCTiOU/haHiN4E6P7LfdsjNwBoidzRttwe8ct/Pyr4j67wxFXi799szZ31bgSAl8MOsbaL8dvo3SI66/8zGqaZKFTzrKCQBC35ZtCnO6hnRrGMcxAVdiID831z20zHF/mEMRrTn485FWFrikGW28+/EaUL9imKYyYFM3LDjt3qdTF7924tWUHitCwqoDKeZp6VpTdJ4U6+lm0sULZkzQIsERhg9hwiJd8A0GOLbsd2KFpb0GeWtzieALYPtFkkcLa4RoFVyGH4GWrqCEyvhidYnaXBx7glKtuyGx2g1Vb2E6k5nh00jcQCnFIAyXkWp6nfDcc3t9SfjVVNPRG6twTPBlnB1gTAAShjITy0cdhs9HM3d6ciJ4132bEN07nAIHcylSa2KiRe52jmO7Ph7omnjyfazxxG2+GtmW4wX/bU/O5C6ykDZrxC00xo2ZwxaMpngDBcMpKrgEUo200ii5bNlBeJrscMLxdGH3t9IaK7jqU4s2B9fumMJgPiB1k5jrbs/wY+ViKdW8yrmAgUxDI4mbzMd/BhEHw3pT1cGMItCHLsLoOAZbSVSC4Za/1p7mZcNulR3AWYCUfUnKmOyx4i8IK4Yax1/ToZyAiTgtUEc2GJodlQVQimr788c2bCCcWp8b7yY7g3O+oIyp5AWJDlSJZv57uBasO2+phN7C++sn20dyMDW7QBAXRix5eXQ49Xx1J8MtHqwpR/vkC2TTbVeXlRnLgQBwpGiIrKIoGCxOM0eI0eAGgrsGwYeGLC69luDvIYJVVoEBNBYkFnpQIvIT5zdh2SzWhonI39fPntOSJsbIc8cm40ow8yQtZqXpLDalUNTf9690mFdW55jI58tD1mKcwWA6uhRiB6n3chqnEZagHUJnma58tSn2rxpttIeCkLzJiAppI2sTQdMDdNpn+kMWflH7NSA9qLatX+MIqCsLi16PUIpHqaLqypEsL5f0KdZQem2NE9UC0KKxVKXqUPO/gaIORYlsBFFRZvJfXT5SvpviaigwO8fd3s90X9jY2FdXyIhSx9RnIsOLQALK8AhmLc+sCJRdc1KHbm4mbLwJyIKpxnzR4dQXqFmQcxJtxwy50WFCzG8OtCLrGBEguRCVt7SEHqtxAD0yXY868yo+dBkILMOCrKUfznpsLCGvfQEoyCLxl6kW/vN6DGdfhorJUGIW7Bs1hWJZI0D45wC2JCJyvfhSbTJi/2VKCS2t6MopcxgZ2GVwQV2RHiA93qB8xV17uhmboPFGOwX48VzCgaWcgTEKwMOGOnAQbUm/2LXBz6JCR/2DsSPX+Y3hoHFFrUa0rX5W4eUd5ZGDWdVv2VBPcq0pgyXsfNC4O+VVTgivYIcuZhIp58TbfHaQHKerF9+A1gmHvvLhtRcIa+lS3M7Q2E07tOBrkHkGgGDNuw8XHSJLJTikW3poOSOzhhDOK1Xfc2uunGN2cLM44K8YVZOWfjb1KcKvLUl/Ga6/e8tseU+sIdUSbxWhHfk/l1hk8EPpWWJRDfuSrhja3xExH62jspPSCkOZQPJutlHjk4bN8dypLneDOvSwTXDpKxElebzGirG5HvsWPjUi+5VdYYYjYza7LQsU3MG2MRrg2+0AK+cUGDSJSCJxCFx4upF3eXoKHIo/TNG0a/FLLwxH9HswN1q1o5/GA/P25/EHFSKargjmsuhdOKCw8rA9vht1eoATRG2XCAHfMQM1QS3Fr76RNTfJvKvCYIqrs1BuoWvhFqSwlYNHuEgupJBO3i6YTeSYMtYC8K8QSY9XYsMOkp9TaI0xW8zUaABuZkUdMNkmvv9XCbr/2rxN3+fKY4sSyfVhHCyWVGLUJfDL7uaVjECj/OlfJk/FTPPASZLVILDBrOwU/Xn6caLxLZPaMddYPxS3DRF+GaQWhCcazmVyRqPV6W0dyRSUjz9NK7KSLxqd9MrJ9KluNLLsZ5kKrC6f614iB0dRTxbF+9ghP7+AbNxsgHcFugCDRXG/4h58fS8t623i44ckbcJX3qPhWOGPkeC4fpRfb4G35w7dCOaAeMB5++PhRfTPPGngmPvXWpuGzILqYP/8qePIkeB57upj8PNeWEyZ9z8MNCOGE3SGaxGnbnQYxjkykDKgDaJ1I8mSwqnhaXJTm7C75nqAqeXZLN1QNjq8kM/GWCPkqNy5xY5Nia4p3xFNl6jJi4sIbjEFbZ2vSSiLCX70cMYL5pPKLqQ9GaTBfDuTPxCxGyUa/d4o6xB40rzIQO86enqiTRU7KKkjweebcDziX8VLtNbXFAPpBPG0bx16QmuGfdIX87pAoXM+pfNXxiuTzE+5QJgi28juuEkKqfasyVsaPZ++1DVGtRQPewyUm/nG3mbzO1EPcDI8NVVObrCUZD+NWl9j0Ge0mx5O6wqjDn0CHqkdUkYPH55jcL9jqaildKVmUXb5rGShEOuwqNbtkxSK4U/J2H1p+2QMqYNy6OcR2BELPEPUgWXW7PEKmYiZhUwW8dyN2RL1E7hZZW7HBLLgYz40HBbzUgPChHClXlVR1cCZLpLdorW1o7Ik7qLx6QflNsn4lPrtJLrb2+IzsxDcp4qEFGqC/giPY73goKlp8xydxYnVrp8fDIw7Fszh5QxpdaLopUVZP1ifANMeHQgmeBhc+SKICWYA6lHXEa0b0CJ5OvPYZXXfbHqYydSh0sE79jWHwJba0EJgJ8o9bc76uLPYubGY/5SDRpeT4TehdweGZyYuFjYXx5qx+tcVKg5mag0J3yfH1rlredKxRzpTqTi61L8z9Y5GqVBt4PGfZ6lMdRTNgiAf2clPi5QDIRzfB84Xo35d5ZV2vIXatwYDJAoN9Xl1RzhIG7koMkLltTseq/WHFUg8rmFJa1SN8GBxgn7/YR5dO7wiqrJ80BHoLFPdq8WxLg+WwOnkoUrphPFkqYRR/jAq33JIIsEvihIenSAVjc2jw3SiWkV2fLk074++KARJ+/P4hFtSooYJ2Y2QfoCbH/f4f9KOgLNXC+SPmB2stHMRWB2tPNGSbpG++SnzK7ni7dE5pq+MUBtK+ECF27+Wgcm5+bUgX/XvF635tMVFd/p9XlnNcTUsvz8hMVzYTgNh2D5W5d+37qKOBvjJWpytzzwVEOrm5kne2GJYg1g89tVaE+kSllTy3np0Pr0RJDJ+Qdejdl1875+cSujq+0ceC7mj4xzjvx7oeR/NxJc2vtLsWFtr+GkcglMmzmlqiz2FrWb8cyqqQ54NMOccj3yTswKLgX0LkT98O4z+cdDve0pU1Xhg/Q2jO34s98NfUhMiIr+HpaI2YF/TgYQBQGVWlgH6//CqdJ999LTfz5HFK8LHp+iKAaGc1gJI4u/Hc5aFqebhrad3YI3s/U+fE1UFKszT73HFXfbY11mipKu/pMaeLwa1zVXj2HRcZ3t9AwYJ5AI/2gNUpvImBLHBiIHlbhDoS34hEwQQMoqhoZeHqrdyxpqYvuxGKAe9DofJ4tWg9h80I1AMLCa0T+gZN+XHYTV5x4bEu4vG09NzeEz/wUp731glecQ0G3zgearqooHimLiYo9GU58uYMSc1Rfmfi/LNK+/EOpAGT4K8Xf3tuHCo1gJ29qELATz2AJ6u3HPfAKccyahjOzGECDkxI1snrOU2N+ecnOAXpgbMlnTPOaRrTdrb1jP305cTY7mTJyeBrbhXSIgCTCcZHlrSShyhXSWie6pnOvYcf8muqUJyJTRwNj+90lx2KLAz2rRGugBpy7wrsuDIO3UPJjs8vckKPKF/UUUUkiRZthuWyGPRRMXPqB91cvhbmZEoIktEJH98ZWc9ZSjTFd/cPODhrtJNyZDWl6mVyHI2WdH8L+X+goo3r8hb2gYCOtnjkMWruFq5X1HE9EqpOvIA+nvkozS1c3GQki2N9O/Ze1AgnvGkxgFMQT5NFxUm3/o12GQXh9ACeBMTkxob7B7wPfTKVozOa7gnZGQP1xKAjr9luItgjL3LK6Jw2/A+PP63FTS7xg2XBBJMOR/Q8Jo/d6yMeFs+kDZCg9E03JvBYem1HecKfbLfeirX8htgwLAqu4VmfKzTfetwQVf51J6HcywJIfl5ODSQCozv54D58OEFFNOCsHLLsvGx46uoC2fJPxgt31O1+Oj9+ytlUN7mcvL2Gp3wd/owvT5F3vYyuVbPmpO/5C85SRFzvaQZbxiUFD4Rh3Ay6tvbO6w58RFZRBBwZaD11B4mDJyZPRxrshPn7qWk3aM5oNG3U8Oo3EiYwVDTyUz0kt1w3+EoK0cjsRY/S5+BxlVWFHWgS8bcBJ7esue76psUSGtBW4CFisRuFTAxPIIhCmg8v3rqG7xQdnKZjkjgNTBI/Cy6y+Xwu//MYT5DYrM0xmXbqKh7ZjMIj8NloUyz2XGyBCmZ3mQr3wbpdQFoEKV4JX/BLodX0Y3N2S48VS8zK2Z7tmpauFBFtlqt1IlbYkv+VBBb6S/ktsVyx8T0cD5mLKebuhOx3LaOywQ1bWgRL/L6kd94PnK55DPv8vMUVHMf8Fu+gdS7etMYPF8EUH0IPutD8NPMUdEoNYfuJkFK8N9ONoYcUHgjeVmv7Dk5DvwIA67t986PIzVj5k8hVsYKtk0wei9ly/GgsFN1SfkhcH07I24QgqY4j8nuYc2a1WbRZ+iilrdtYGI2bYJb2VdTyEj7PFIwLYtxOtuDGWqpl0uyAR7yy7UXGL7uLvDe/ybu5VysdAsjrzPgRIHL6KZudkI5cyxiM051nOWzHyrxlzTpNwNCrQ2cP5BQjCjyPah4OSgLzDiDjVocREgvrzqvLHKwSRLLMOaNuDDPy/e2WMnQGdc/9/2gq3ayyMMaogD8e5RlvILm42b3kWXmDXkkwu4jTX08Mxas7wbHT/iumxk2anzvP5vH+w99q5STr+4MDTUB+2wZe5jhC5v5b11XgIBXAu2lU78PJEg8xNnnttIGzO+Q3kTV+EuQ3Zbc0rxqgW+DxALS4EF4Mm2hwia04uAADpCuGnssypHUDRp7qJIvyirY9l/NxoVooV1goZDOiEeNEVHxbalOKvnmS9HWPx1IB+ge2ISz0MdIXdDR2RpuJ+oSBONAk7ltnbdnAlDz3u1v3HckF+rXYH+EZ56yUg+tzHOfvR9Jf6Yri9lbcl2VffWVaGHGnfFYWlKQ33vTAUVhtCg+zto57esZ91Qux+g3fR802k3dmuhCMJvum+bQwtB1dJeJFPeBHSWk4cyIFyzFfirGKOJMjcBInGk3Tkt+I6zszrBrPBA0IvXFrSSHaU5wmBwAtD8NBAoMZDW03JgtuFmeCxYeyHnrmNJq8FUxwFMRBfHLea05SvC+/OK1cropYyX3sFgJbzMblS+vJejo68DoWAJlQHr850RlFA0+H2U+c9mPu402ko4eeA7weQaAsmPfNBMkd0VC74d63nlPEY8FRJPa+dUF4hEoB8LwbMTfv+gzU1RY64Z5Yip8zyhuwNvK2xmYwrwPWDMhfIUnr5jqSP0SSDv0mTsuuwVO5ea+Px0wd6BRX/kZCEVN4Sbc6ZKgGjB1y9UHvjmsdwdXC+G6pO/fexTt1E+Z8HeOlBvLrBXwNxZLkT567VQvmrZB+dHX7e1tto4lQFUGsvtKOlLPkgZYPYgm2+3eYYZRDtfwiahRZwCglJ1sVpCuxjWLPRnhgXL8mtFKM1Wv2aVAxhD9TN7Ejuw3xdLJRnqOsrKGBtQejkneJTtjdiYGtbXHzcmssVlFiBzP389JyLNAPsINQ/qMBEieQHDGqk/YItUYOF67CdtsqhnLdITvYK2+6E3gm0MWSDLex+2MTYyU/Ak8Ro32990KSk6dhfE3c9A+mHU5DMeLNM6DM8fgN4CcQEleET6PjhTBCZgKMGEk6nH485Fs/EhN9TQwmASj5AzsBa6boBP9Jg8wcLe92Niyr7m3/mof8ZQwqUBc/zrPQumPp9y0SSw2OzsHppYjHizy1BHy5yTIBVCXw3rzhQVJW/NZThr/dgHt6AkJHSnXtHIqzR3+IJho7+AU4WZjiBgwZplUlVMm6RXDnDnQ/unBEKEx5zQ6f7SnbLDl6ziBbK98sTMWHoNMOZS/1n8dvck8LkjVRryN9WvyUe0UioobyXaw45XsZw9u96MfehMTNTvtf3xvliBPjxN6rGEm+0dk8I9lmPLtnxVCRKZS9Hp1wgO37XcCbq7DIDW/KXtK8OQY418jgFtV6Koq6d39JDPDEOHXHDSkD9Pj+wn7AZRfipuot7W7y464hwuMg6KK6KKT+BT23kKXXYTvUvCjKqdwfKRTtjXqce+11JNbCSASm42vVcKGsfHprbQcK4rFbDf3qZlMNhRmQP/v4pboCQBYh8J/L2+NvZVRVc40bNvKHnlyASi4edxTtqxsBCEa+2QxgPOiCK5rmTF3OT2vLszge7JH/5165U13sSr4vjPccSOTJGSsE4gyvj+mB5XW04jesjw3V2r5k3eD/aiGQWa9jz474ZdVsPvGKhX6fgh6uIn+E+2xyJh6oQL8Sy1GKrG960gXjuT2ZCKWf+iT/iUB0PJQ446yUmFoqmEfDH7w6FR8+Cb78Zj5XitATAD4JvpmfnKAAqEZMDEq6ZHg2ga2PMacP3fICIFRms4s5Cjfd255f7TQhl4/SL7edSUwioXjsoJZdzOd0+tZ884m2MtUMl48K37lcr+QmHiYm7qCJh4zJ2ToOR4lOVutP/KTkNvwobCTXynTznfodSS2GVtzzfxzWpKA=\", \"train.py\": \"eNq9O2uP2ziS3wPkP/AENFaatZUOsru38IwGyCY9s1nk0UhmD7doNATZom1ty5JHpDrt6ev/flV8k6LdnbvbC4LEJovFqmKxnnSSJL8MVdMRvqVk3dzRmrw8P58P/djV5M3l38n7ZrPlP//lA1lWjLZNR8nXhm/JQNm4q5YtJastXd3s+6bjLH/+7PmzX7YNI7u+HmEOBmnHm76r2vZAVn3HYStGup7sKr5ve942S9Ls9v3AGekHgkO86TYAWlPAliQJolwP/Y6U5Xrk40DLUq0gVdf1vEL0DKH06LDZVwOjZuCfrO/Ml7bfbGAD871n5uO+rfi6H3ZmgG1H3rT268HC8mZnNxjHplZE1hWnOKdJ1N/V9L7iW8syuYSvaoYf9si3mnjdHWbkQ7XfC1LNRt242x9IBQLcW6qrroYR+Luv7SDzKb9paTV0RpT2yPSGb8zIh6qrNnSYkYr3u2ZVovTKGjaekVXV9V2zqtpyWzFNOJw0bTWa9PkzAn/ecTqIc/lMV/1QAzY5LjQNWLqsRkY/019Hyjit1eRybNq6BIGBnnGmBnfVaujL9ctyR/nQrNTobdU2KNmSK4QlqNa62cB0hkw+f/b+088/X3wmhT7vfEP5e/hIh7Qsu2oHWgSQl6///uXibXnxn+9+Kd98ensB8P/+R4mgpmtYW9VyC4U/xfNbEMYH8l/i8DIy/5HUzYpfwdgMj+16IUmUC0pcAFgRVizO5Ky4Qg5I3u9pl9IO1B6ILZKRr+d/TjI81S2cb0sVVvzTrL2VbFzDtc1XILV139ZpRoqCJDkeW+KssjQBOTiZI3epxJ5ZONoyGizjwyEYEWTIEz9Uu9afpHcruufknZi/GAa41sAGjEaQgGwZJZ/HDu+IgE2Tf7z+8F6ROkotAmPz69iAxSGXB5z9nvzty6ePpKO0FraE3sEhkZqCDGuQ4QEEJ1QT9jwiAKQ6Z9WallMpgHzBrJCGgaXiVbeiqVIucdCZw4Wk/j+qdtS0awUP6e8B4W5knCwpWC3SL/9JVzxRGyruaqDr3iJP9kOPUEJbkxlJxEUz3+jdng4gtI6XQ9+KIQbywP9retusYMTBBYajXPY9Q2Aw7AJBNbSHkvFeGBkcaXbLqkWGSyEPNbqmlbC6YPyBGuDGQwz3uATbrm7J0H9lllSwwtVOfFeXGj9a0yNJZkyghI+v8F91XfUeD8oKNADW4cEx0Clap1pked2s13Sg9pQye4pq1akDWycfFGoeP7kbemALcq9QPegjO2aANBH6YEFyncJozYpiuqxphVLWlC+0wbe2RJiXNagoB3vzse/0zdxVd80OjnTbjwMDqQgQhebKCPX6KvEAk2tFFx56uQRzAce6a7qR05NIIuAGFYrZI+aHgpy7EpciQNrl4MgwaABtgo1q3NZf/h159afz8/yczKNUfkf+BJNm6wBZuPfkeiqm8hhqfT1bgIFgCG6phvaFqBjXh+dw4lMjwcA00QGiH3PYANeznHa3zQCowSulyeW7y4v37z5elF8uvnx59+lj+fbi9VsxcHH56c1fEyvqCTaH2whBwFoaDs9Q4CkIcaYOfIITlE6ENzn+k2aZr8xiZteDhcRQALzN7yc7W00310REiwAA0RpNtRLC54jOz8gwdmVTCzc7I9KIiIBDjRiDBKZkV7kzHEI/yhcYdUIggHcHPml/jHaHBSZW7pQsSCqokcehBjNNSObaO4ecYJk7k3l0ewgi1AeIYhBZlG0PsWS+bHTohViBfRfzBGRG5i8zwI1wcjLLfMu7rpp2HIR9uAdbCKYw6ZeMDrcUxaY/Kn+04mJUf3wgcMHQgs5IaiH1bAa7qlPJgaIdA10CFdeA5N8Ki8hcAE3OaZv+xglvGWyz6ncQKjWYrIjAi2OGAocL3GiExrKvxgHcCQd+jfCuEjXoSO7aDxbO0fbopT8UrkCfTqpGTu4Vpgckvx85a2pKzvP8XqJ8SAL/IqHttRtAQaRXEt6epRP6xR3RFyYYDi6OQUZctuwaNBc4PtnDSsii+OERA+2Io2oHsCsHEU5inIf5qfbNoB6SDCu0QCZmSyuVUvvdBoNk4BFsxsJbtNYOorzXeVve9V/TDBzGsBbm8Hdn/zjbndXzs7+efTj78rvsobzH7C/Hf/4AgFt6d7X48/VD4uyrjL2I1SCNqTAaSltMqzfLXXlLByZEL0wYBmp8i5wzexjRBGNHd/1wgEORuV4Oe/ARzLgcT31xeHHlgW+BRbUv3Fed9+b+TOobPjmHV6lp6RBZpqf8dSGfsDAccsFVpupAq5G81GNlGUS2+4MD3u2PQcpE2eW7PgqqWPO4VB989lb70QMa+hWoEKhyJssZahzS2C1Ger5o+o1IpgEHCA9yIMQkD9MMpQqo+GUYqX8g2wN7+vKfKsjrfMrNXSodpUuE1qXOiO9keg47gmcrlwcMAyW0VLlcTDr+w1yAr0ODMcCo02gVXKtAAMfrZliIJHnmZs+x6EDOS+/62LwTGyhvdix0kNNttYS0ZSdxBsiBz+sAC6ZiDNWZidupp2Xy8whQ/PqrqIWP+5ZeCXEIoQQ1BRAWXHslNvJCH2USAuW7G/g3BVmA5WFCg9D9AiVlfyMVSqcTXbOmjLuVAZamznawCat2QFWpYWWBIctRR0oOQeS0eqGwG0XF7BBTuHEnAhN0+leJNwtuVcQMOIXhgd4MchCIvSExXYOBcfIYkZKCAgeYb9GjSEziY4Bq3zZc5DXNb4gsFzAQfmRWH02hAE1vkNSZ6Xzco5tIoxGlOp4cz9+9QWHeviCJLqwiWg6DJzP3BZZnPYR+Hr8g4qq7EJG8HnbtICVLopGpSfRjUJGEf0GEakVQrYdKY3qZe1QvK5HnPwmC/gqz3pzwCCK+AfnDpMxk5KW/CmavPQOmKhElGOpb2qFQYPm9XyUyQFi2gWmrOt7E9SxYJm4I7osWwFvnz0wXWuU2lrylXXpU/UME/iVa+Fduspt7a9DBut9DYDxqD1sJCiAypmkx7yi/oiK5Hts2IaCA4bUtfHJ9vBNWxe0t5dVdyOt5/GpPVoNq3FQb0OCxBiz7iklm4IK1DhoPCtEpQA/fw/G00Pn2eOoXGY0tcvXC8y3R7FLMOMjFdxcw4qRQY6ejx7BrN6kP4bdm79M1i2HLstC2rVqULVOceR7Yg3WCaIB9WkwtE1ttFtwo59rmuY5ZV8V63+859l64PC/AiK5wq5/umrChklrUM7N/dgRWCFnitNUFZ4HOexQaD8QJxVY9RMv7gWJQUFvJ6REVhPkxmYhJRADz/xGMAIUNEmI3BPxX1yr0ApeOioQePU2QahlwY4dHSFoVm9G+o7E0g9HoBevNrsKpmMbJUKXdxSRLCQg4xP29bgwWACRgLhiCUCJodchE9ycg6GPPf0I/rvP/z7r2rzeYI/mkGnizBs+IBQBdxib3chOT/suqH+NNJ8sGgR76dMpeZo7nj4JJJa6Zi8BBKw8hB7HQrk6nMEBsP4AZ1uIR/a1SerASg0LWt7eQ7uTyvB29EFdKrHavhsJ3ghu1KHIVQ8482mYR1J6iaR6PgpkiPkI7NwlVft1g4VQuhSNKTTiNHdRFpKlqbY0ucB7PgUTUh2iwGSyHIOXi2qCyyYTQZSZSYMh0h6E66IQC/ZtSSb+ZJzu41Q1VTOiuHsWYGAu3kkNh0ZBz0+OLzAcRgWIp8MCKqWDU4ys2J1kLZiBiaJfV6qZoq92yroi0eysgawPp6EKfQ84O3arUNyqVwp8F0K6vySbtCzRPup4vmpsX4j/M2yysbDnnX2XwGUgj+QkFpeUGC18YrRFFTVp/TxxZymC/hFA/53dcRE3OMwtV3ULvBhwls0n/tWy6dV8EUfmULZEpWJUWIX1aDRvQIf2IIv+IbntfrWhYT9cZ0rRJjhhyL2cCIykGjeMRNhovdsOE+XR6Wxa5ibdFCBYuxbDyCFZ/0111p+qgJRZ+dRnwyNaPrQp6TPEy5nwOy+UbGjbH5XO9XHeZ9j1reHNLk2zKtd+/i1FheT8CoIoM6u0DJJjN6o08HLtdS29pW2wgBOB8SBXsDI2SaQLqbuw1xsQADXETJL17rPG5FwZLXBUvkrO0Yissk2aMnKViBToh8U1+WMAn0CcG9zJjWnGzmGXSz3VUSIdPCNrN0ruEJ18YxF4WvMOGftsapD8Wf4Cc84c/EtkKND1g7/2ARNmPfD/KxE+7OyF/NQ5Kl3nGH4Am1t+RvYOucD47MrXd/sI9EecRgJeNWOugbmGhVzgNdzfxZK9CQPYqBIAEeQmWqYRUeRiamgqLYm7IuBdX37FLJe9L9ko8IRHXyj9faXhFx1faZRUllHJCylO13Dyv6C6RI6kHZmIzdTBPsx5qsbLaaDDT5FKAoU2md2BZ2wMBLuwLuLcSoYhCabXaSoV5obqcaNTRWRGZhWb6VRM+LQHS/OdNJuKeEc9SqraGbLVoRsIa0LWva3ndf+3EWXhtVk9GOE1d4Qv4CGCz1rBgH0OzHLZ4AN+5nQb7w4XjFaG737Rwg3ixTQf5t1Bm/ZwK9wIg+1IA/zyWr5hMxTGjj5Sbw/iE6AxMnlQu86bgq0zL9djplN2eeu5lF5PlKk1Wo7HcG03ekR5F5plfEc1gUqTex6TThCzLN22/TJPvVJYT5ihPiZYMLmWH/bdiQptmWg1UaIAx8yn9OfVEIECsafnGk1F3Kio6q4ZYPuaDJzdr2ETW0lZMwWJM5jpuz4Z8Rj7QhAAqcsakJ8HeWEs53irD+xk+w9I8xbqn0/7p0X6uZlI8/nPjExVv42GAKv1FfkuVrkHYWXwj05kXKaGRUjvkE5ogRsAmvnqUEQuZXjNGBwRV/vkLHRpQh99ATCIWJopeUvdUBmvqBbExzxAgW4ky90yMOle3fpZitEhlIlpykaVY6QVoVL+RmYWJ6EuXcOtU0pREcJjovtRHD0dwImcM9raKYcyUoddXdXaVoNMB1+0ZHTMcVfsjHE4NpOBXc5BIXZwwJvz9calEKfCvjLwpX/EZs+r6G+QVP3ZlIjIXWc25vjm234+yd7v/vvs8es5j18lHkMfeNjhRl3rO5Hs9fDXzoMnBFMtZsO37m9TL8c02Jx3pjEg6hd+0DwSdC9b1EA1AcjwhLxouKuPqjjhv85xLZCGiilIouQWm20i9sB/jNYEiXhtQwigC4QRQjjMoIv7BB454ieLJrj18/1LIgdnUZY2skP/F3XbA0NX85bUbretjELnedHh6mWU3UqMrqzVQ6B6acTXTs/cpebopVOw96gccjNPH5MJB3ja/6ZRvQztcScumW4HnA1zGXjK3BoV/TsFGOlTxmtS/spYUnPmJ8tGjJSRRRnpn+bQ/lpEFJCJO3LOW37u/9JBRBJMRVxKRwWOnGFlyrNYUrzfFtFgXWeXvREDVJ78dSScZRjG1AY/Zmcdvrdc1LmRH2bMkJxvLDmv4m6cBhFy4KZybhV8lGkTFc0dRoY8ogu+zyJNf8x63OPK23MV/pGRUTMpP8tHosRJUFpE+2IPBFfI0TvXIEK+q9Vrxurp48kt2d/fpa+7iGx+z+5UKbQzQBV5p1VSRFL7e6OsgQXfrZLoHKQHdh6Jq6Y9egG720o0IjNMBX2lLxalcWMj/smi5zI/zZSU35iF95xi6NlHFBVuvoeSAqlkEwEFRoviGJCXAJAwSbsuKq+jGOrK1hRY9dR1FJZLnAiJgxIJdPrtw8vhijXIugp+ZpdiLjuXwEyNoE8jCfgxgbijd2x+K6GgnYjGNLhTm0zR0UC4k/lO6aR3UC7TP2Peko3dcX1jytWlb+VPOWKQNmGaiLa8DA/J78nJq1sNf0WmjLjMdN9l/zMPYd97BUpNH4qtKj6JYhhnNLoNjtz/VklyA7PBteD/yI8nlwsSp9wF1D6bmUdy7tD0kk8OTK9Wrgf9hivq/TU89jXhjChPyt7/mR76eNoAkFpChASbLgNcJ/rZE91+U5J6Wj6Me/6eJrd8mPrftNNE/K7GqLZ+bT1tqC/OWhAnTbSBeD5sRg7xLMYP999XQiICxKMu6X4kfr9qleVXXuJFYg90nVabD7v66GlteqMLdC2GFVAHwJAbvOcIcC+YWGZZoT6+WLY1gWSJH2QuQP3tkewCZY5D6DXse6bjhaR32tBC/d3g6NtncmDvlsDnv5+I3ivLBYoGRxIA/ABxp8NsDhdI9f6sV6CSlPjjFdWmwvhxA93cXdw1PpQN3MWQKB2iq/gGzeFZXloixLBPz00DE//zZfwPkbWUY\", \"viz.py\": \"eNrlPGtv20iS3wPkP/RxsRgyQzOSEyeOsQrgOM5ucHnB8S3mTtARbbElcU2RHJJyrPH6v19V9YPNh2zZ8czu3HgGCtnsrq6ud3UX6TjO3wSPElGWrFqIMi53Crhfs4u4XPEk/oVXcZaWbJYV+Jx9iOeL6q9vPrIzXookTkXgOM7jR48fzYpsycJwtqpWhQhDFi/zrKgYT9OskjCwl2pd8ipPsiqJz7CxvgtWpXCdw/nc8aze86m5/EeZpTaUhbkpF6sqThQeOTwBcBqJL9SRnlTrPE7n+sFhuvbZEU8SfpYIn33kOT712Vfx80qkU9GLcpCv8YrxkuVJZTqkq2W+xsY0N205TyNowZ6RQsCCM82SrCg1Lh+y+aesWKpu5XkieJEGS1EV8dR04qupz/JCTOMSSBrCBSAfTlfFBaBfZFN5iWg/fnT0+cPnk/DL4Yfj09NjNmLu40cM/pw/DQYvd9/sOj5cvt3bOx4M6HIweHX88hldHh29fHX4ki6PX7x6hx302L0Xb54fv1ID8I8uXx4+02D23j47fPVGddiH/3AscPPD+0/H4dfT//5w/BVxcXawy478DfD3AHn+8fDkP49PZI8MW0v8+V/8eYs/F/jzBX9+wp+/4M9r/HmCow9/ev81fPf502n49f3/4JKHg8ePTt+ffjhutg5xop/Co/86+ftx+OXz+0+nOOMerAZgFFU849MKpeKMT8/hgRaQ8RgFyWdlVUx89ilLxURSOhIzFqLahCifLkrfAQmdx3Zeo5AdSPJ9i6sFyWaQ5SJ1QcKyCORt5Kyq2c6+46GgLEBkEqEG4F8hQKNSkvwgyXjkyh6eNfU0SytxWbnFKg2juLDmrlY54B3F02oMWPuIC6CexGXVatStcm3UTD9xWk0UMggdZprFc6CJtVw1K3vKHPnYwcu6d4C9kD0IZAGTZMV6IwQl7wRC9W2MX0oV3QYD4JlIQjWgAWSa8LIMU74UJQAa4wUZOLzwGdiulJWgbSJy9ei4EsvS9Xx2LtajhC/PIs6w7QAJ5OLVeDjxvImEP4tTnoTQWpDlgzmW/NJ1sStobFZEY8c8dCYezS0f4NRq2TAZMJevkmo0UHgrUXAt4TBU9utGBcBqsRZstQKD3RrA2FlmERAMewFSG/sFc1G5xN44As1TtA9wmGePahFBPbHFtoqrRLjoSA6ksBEC6lpOoG5asIjqIKCr2Sy+pC5AYschiYcbJa95kc0L9Gv4jMWzLlvQQAyYSEpgv8P+ycB4FiKt6i6jq9aYa0fCBm0rOACmYVcSkWuaRF5LoI7T4NvMucLFXuMQWipdyYXiZWuuESB3pRdxfUVTXjsW/cpqnYiQX8aliz8H6I6Cw0vgMZsXSLuzLEsAydNiJYg2aLIUcXBAUMXT8zDnBV9KCCPnLKsWwFRSnTL+RYyaJlXJIcordUFxJUggE+ElgpMjXc8yYNQUlNCjUKGA+3zP63u+4CBYGGBoRQV60krqznI2aHNxVWi0UkF0GElnwpN8wUeD4LktaNhJUgt0MBKXSn6WHCyzJtI7DhwjKjUN40GDhVc1Jg65b+eANfzsmCZgf2YJGPjGE29iKYdj8AYAlmtsDLfavQnSAhGWkmUPkWtiT5/2zOn1gbLxWPLiXOAqlOdtIKDa2pO3u+52IV6IYg1AB8FwtzsbChY8fBbstinyLY5A/g7YMHi2p55dW3yEUDKercm/grTrII28MjitKa/EHOyeshlT5cAPWMel/5M0oa0SsEYzxjJjIOs4IYq6nLh+RqZVDSGsaiRs8Sv5hQjBckJg7GoHMZfa+o5a/drBGvet2tCsygXJ+woDERgbBW95xd8V6LGUU7ttvSDk+I/uXl6EShlsDcBgDelCwQAR9sCgDJiiMbW8rWpUZk7Hqo0u2qE3oATLc3gMJENzW46kKgtQ7SrMzunWa4Dcun+O0XsezXxaH/FtZFB/ihYYCXodQD/H730QzeCBXon1AODVawDLCVYqTPg6W1WuV7cjq+Ffl/CI8nj0bDDw2dlZdgnEnkJ+NXIq28A1xiDaPV0JHR4Bu0dXzhFEPWhCgfuoQ8RQ5nzMIqvh2rNkJaiyEHB3NT0wpAOujwz/gQwQVlYhCDrkJiPnz8GLmUYPRXSaZJCUAYKqbT7FxCURU7NwrZeupj6ELkY0akVs9zYYYXctJp3uyvCOu5y1g+8o+5aWfAnBrvuEFwVfg31I8wCyL7yxQuG60WdBEGjpBms3J1lBsycBjAeT2gup538ZsXbi0I3UaSIXJuIlQXIveIIOGC0JXZLXpDn0BCn4HNItGLVKY7BrOB4sYpnzqXBBhBQCO2zod1AASYO0VoxgCPi1F8+9Juk24DNWs042IKZpi9lqSMkoBMIyvyybmYa/te3pNy2hb0JWvKbQSAeAfk8w3c53bHGHx7ZtdHUwrSny8youRAS9bE9eB+OYcUK0hbk1RbcOxFxxGi4hCo/DJJuDLlBGCuRqN1rw7EGiKCBKaAwxTaofnxZZOBuaTubeeEAyhTHgRZmPyk30YoIons1Ega7QlSoP+rlapqVXy68abAsrj8GV/x3Zfoz4uLNGvgVR2fS8ZJrvO8R34IVIIlCtKwXv2mRUkJTOZTZVT4H5jkQoTrKpDBcmzdSn6WdlQAFCWCA0d0hiL0F4VldYTw9Yi28T9h+jThfUnVY3CdIkbWBg+CUtAu1euTpD0S8RjWfkKyggdocvfbYX7GqEcp5CUFRvr+CfewexAeONDTs3NMjQ1N8E/yYJO5wC4/h03bomj9kHcKMoAlZ4vfOufd1AzsoPMFD3mSvhgkD6aGLkBZHMZ2sK/NEfQbRYecj3X+Lc5ZTASLLaaYSERHYKyT0MBsBRYvLYTEKBqoQnQ9X28xocYtMLTKG5AZR+2spIUFKkoDbk229grfKqkXOKjUC7J0/s1GRgy/jNUGvkDUzQ5DjSRqwJeOj16ZnPapsKpBfpaom3wlW67LUCXcKIX14gYLdO6xmlQSPnTy/oz+mmZCawHwGZDb4nooQJHUllVE/IxlUuTp6iRQpMD+V+gZKeGfgAUsjW/l7fwEua03XeZFlZ0favZfANoN5UtwFHSqyrBXfLkYmYizRyTe9XVod2Dm+HhatcLlit2/mgbDE7Ih/s3OYpvVuJRK5du03tOvK123k8VvaBaxMyMUrT7WSboEkXEsruLXDahkxDwfQL/b+dUJHZVoEAWKZWoAJEqgF3Ykvl8DA+7k9a7F3IOgFoxNUWDB1ay0TqhrDahMD24K2i4CdEA9/GfNIO1YqwhPQhWoHM/O7CtPG4GYwZdqJlciZNVx13XLVx0vvBXsNL32KkZVtrMmOqjOJR+0az/RDmpm1qOnNvObzfcuycABD2VUnHAxiQDeZLSeuNelpLqaMY0BD6hkgbjGBByweWamsH/bcX72auUS+zFAA8omAQiDQ9zzOIpU2rVgM1GrUAl+XixmWAp4fnYl1qQbfjXbBZ1NHTIqtOICEbUt3usJtpRxJqdDOO0Oh519tq7XCISvuioVBnvLhJZ7s0m+iQZCxXN6ZAqZXl9hAHxoloLlQ4c4b5TyN4GQS7xiL0aLZJhAy7iuxbqf3qWE3Yx032mg0mtv5OeQVg3Rao1vo7T/tA4x437riO6PRUR2rm4LYc7e6ZNR2Z8ewMkqZzvSLQ4YLPUaTJobmbCR8sBU9BuNSGiUrcZEw3CAbaNiwhSJCJLJ6KQUSIhzA7HeOjZJzPhJ7BoGAfWikWOpDaAbh4uVqGi2xVlHiu9oQ9eyFh3zQMVDMPzwTIhwiXcbqqhBr8YtDUszAVIqLNA6w8CKYiTtx6OU8MpZ42sCZy6Ec8jZpLeq2D3sGv6ES+GnNyL/fx3lg9RN+Sk1Owxrd6EWhwIB+IgWZAu5pemqqjqxZ9r53bHY8dWNOWKURjPJ0uQL5diCdBU4cABMR45KzyXBQgj7PKXv/+d3uwplO63YnJM9gI7G8Rn63oGOyhwzP43z7pvbcPW/I0ngHPtjlll7uuoR7SOGdXFtDaErpyyjyJK+eA0b/o3xBhuJdn7wB5leJj3DvSMMdqEOiqXB51AiUdU/uETqTleAjGrpu5LnVBk6/2ONTWRp0rO6DvleOZAgDsa1GxsUfU69xxlWrBl3ILl8v9K7SBFiTt1sidQEdwKHvbeka0lEO5DdwACVZnELzcBVk3XtN2y3rttVO+hQp20m92R2S+gxtp2m1KdmDKbthIfMMII6RCKLedBaMfv2Q/MtdsxiHyRAuf6b0MdSs9EoEOpA3ytPfqCU+2cNx7HcsIThZNW5LNnU74jufYpXvZUKZNMf4Rie/dLTJpTclcQIARLt59rTNhwN5aduUBAvsbNy7ubzO7NvBGuxmq6kE0LzDislPmIJvtox6fpZhwJ4CvKoBoH/OSpEofDmODcsFzMdYBGJ4B0/PXbLjXOdpR4QDAhvEvG52VF3+l7MSiEOUiSyITsiCGPEU1lvNClPSU7cpzdYkHQm2GSlQWlH2rN8RpNq91OC03+2/oZHTZLHoMUH01cNLsiN6A6lquaMhB8Hwm61pqskossSgIjbQ8TPJ69grJschJsDYRZsQWyLL4yJkK8Du4Q31h3xhJwx8Tr35bgOASDnIVry3yyjIbqfMdhwtOakXhPa162nNspXs0Jegh/PB3emASLYWcFFE80iCPIs2RdAXKr9x0rOMcaTiK+UyCk4dGJaNq082lcgX/1vZ3NWK8xHNH15w76g0wtXh1DDWyDSiVQMIAURQlajYkrRdxJEZOPAcJwzAqTskxmRZ7dbUQ1nv3feiQzgFCoGMWGVdLWeU0xDI+kUfxsrRrB/qAS7UNpVdzOz2Qzyk4GPDMeVbG6Uxep2Kurjug28TsAXkrDZUBQmNC+1y1JFA4sLfvbRtWSD0jeDhyf98cGS5lykeKDHRaZN/6UJ2CyEA2jDeoy5A9jQb4L79EIvMyF9Nq5PBVldkFD6TXGA/QLIjiyHIjvUbfZz2zW7xrO+86AFMqojSy44/v1l17/S+FwM0WYOf0fv4fUb/r2KbbP8m+7XyqxarW8o9EtV9hc0+f9akyEJFSRBndsiHfssKXYc1Ix+ZqbXb9ujZpg3rmtHvQU4ekcxN7Qqr16YKIZncBEc0UCPmaAR2TzOJ6h/02CNaqVe1Rcznew0GnAqbmStvHDs25O71vqt4xbA/AaWF0eBsw8srfaY2WvDwnQQNzvOSBvA3xnZVQ/LziSe2SMAPcbMHkQG22LuIihlBUUWuk3r9wyYoNlRWjvIuq4C23h42eh0biQS2ctQhb0f6Apg1ijrY9Yy7wh33VmdJvZNu0kN/JwEHIhHXx/NsNJq2OrkLQFV5uPn7s2KL6HBJG61NIA0iHD5bhNGpfd9rurNGQwa8RbZ82mneOOsH1OqwAh2ZkvQ7zIjv7t4u2cQfnoPetGNy9UhkSFp1LE2TX5YFx2B0MhsZQYb0MBJqy4y+iyMowic+Fi6MbnXQaOtj6cGSAsd7L2iama1WvqGK/1+xFnTfKTFvtA+ltrXobSA6xImvJrfAMSEZv59j1gvKZh9s9FlivGf7vNwu0GvACMKkujW81oyFtZarAwWav5uNu9akpxAbWxulK2GtCaQunalPOrAZbxwe+vZiJVUNJycOzXbtmJgd5qOhHZ56ksVr23QbCvpnYAlFOIZdBd7Sauhqc9ThCBaN2uoKHKK1WVWvPmPpMuzta76Hp8mXmXhEGlM97nTPshrSgZNkJPWqHNoRX7f1amBVu5B7HJZqVympYU4MmmWmuvGuSUZgLiFXpwq8+Ktt41Mr1IzE0xl2DnDTLZz30sfUMRgwtardlrGaZ38tsv4dBNbiuTLa1CRNeDFSCbIUne13d8lvbxspXd3QsKMARJfU0mwW8R6Cbg79TqJcxcWOzWN9VpLcT54847Y4+TnOvLDSMbHfOOTdXpO1aHqAr487Sns357cT9AQW0K5y052hU46CtMJJRtbI9tXt3tNHiv1REA6TXUrX6WHzlLb7yzXztrTQMevl6E2/5d/FW7gzVnNVsbVOhI91jjBrA3+h/9aqe019zVQet8sm6AuGEp1G2ZH9dQRLkbIrlKaZlX7IyruILwU7uVELUiOfvCaMZ139Oxc5FuXOCJ4snn4+2rWPEA2R64V7ajXJ0hRZSGc4D/3dyXFwHyliT2Dk/3HxunBd/kAD7/1sIDGlXK94FfH8X8a75wITP5BcmWjFC/wcotoiCyTVpkHRjQLWCA93JPN8c+PbD05ZSxql3CHdzHkUiChsLRnKCXQDrYbtc6QK2jZQlmk0PY7C9ZwzRoZLfRd9e241xxY1c7wHs38TO33t0/KuowPeL/51Evxks3y8wvqc2bBtP/6uV4rdWiK4ydIK2E0mSO8dpXwzZHiA8M8B2JDp/rFjNhFx3DdVEocr81OZqO2QzHVpv8D9IjPYdRetZEQnM+gx+Ab5jqg5a3bFTAqWhwTH1ehM8+5gCi/DzRWP5Pj7+TsCMChQtcq5gpLLc3oped+rjaGatjQuBL7zL0rjndykax43EF/IgXAKkA6Tn+55dFJfNZoCZPlJXRlIZRSoQdHckApgL0muO9ILjABiOO72WXcJvUFk3KI+uGVpo/a01ebdRonHGqeReV8Ut3DX70SBH6I9VaY7PNFRp0G2MW5Vw1HaHSjhNFEDAr8vi0Xwiei08el8FJDHCjV1pjNVL7j9i+dJg38e3upAuAI0+DgOBLzzTd3JZLtVAocm4smHImqNNpUH7G865wHKPARBo6PgHktEfJtcQwo+u6KtL2KqE+IcJGiLPoeWHvq6wogXjd54KVHLXm3Ttc7ykHf9gsLfJdn+dyhqWO5puWcLnKgTvXQn4RRQ7EtS7IVDWiChcG1H8Ny0MrC0naR4xY7OdnQlOXxmUH8Xj+OL75i+obP5iSvvbKiR+oZR5u11y126547dW6C1GG3Cn169k16ssN/W7tkW3V2pbcnkIGiwEj9xnA2+rMZZ53zaDt8w1YCiNNUTg2lgTuajwOMvHNu3onWy7gV5IaDTguz2waPMecfNNogXON3aU/NBrRHBvrwyaLgHgSOLQb2sH29hZlN6cFr27wVpcbvUaccdYvFPI389IoMyDi84TPL90QnRTzPHqKut/2RuBMkveXAusD6Px4ziFWIJfAMRgISChVpgldetB3gvMMoxEsKm9maXOx2uf+KT/GykAwVqK3+lvv6h7S9/2G5C93Sc91JIMVOU8bStpUauRmeOUjT3ERkK94Wl/Ma3KIjfEvbpUR5tx+Z429FJfYpUbhfaQyd0t8F35a1e2/aZM7itYlkPqyqDNo+1KDBplMdO3eXcTEGuz+f4wulkQwTLtN9Kg5y2pB9INOobPBX2gzP76TzeaCOf4iswBc525elfmNANzw9QdGQ3/Zgj6NSdXviwjT88Q0ld6J0ffdkABkZYr+ZVECxwBwrcrw0iA9eOlsL/CwuzGsorqTg3QWPfdEzrVsPlZGWIngg0NDBrAKeIr4387/EIbztZrGzbq13WWJe24jBZ8pn0cuVP6xAtxQH9P1corOjVXmyI9yd+OzaCvuE60H2lNX9dhSUSaltxM/fjR/wHHOUgc\"}")
expected_hashes = json.loads("{\"checkpoint.py\": \"a4a5dd61b17038baa33ef078d627a74572ba5477255d110c7dfa0a9dd1d1c4b9\", \"config/data.json\": \"f9194a65fb8c649dc15fe469113cf4fe615c5b7dc8177d8f0012150585d515d7\", \"config/data.smoke.json\": \"9ce6ffa8e094991f310dce0c7dd08d8fd63d8e873bbd059230e7c85c53854e5f\", \"config/orchestration.json\": \"0884f91298f0d8dcc32b5958b316ad4150c6efee8e62107980dcc7876f1a71b1\", \"config/report.json\": \"847057bd82f6113f47ac60da728b970fbd8b97b6bf9a8015a087992568e66bc3\", \"config/train.json\": \"34781ca70802bfc481c703026b7bc99cb1dfe543614c8687a4cd2e443d2d095b\", \"config/train.smoke.json\": \"992d3b9e1338522eff222e4b02e14a0dc24db72f1e1421d71f2433ac6880793d\", \"data.py\": \"7e2bab4a9b899f04a788a504541dfaebdf71e2ab0a465602edb19e05b82ff2c0\", \"make_report.py\": \"746537d648ac74aac26d7ef9e04291cd5034bb202c1cdb4c43f3139d89209f95\", \"model.py\": \"28b35476cc49a1d27ab3f30378da6180faa89fbbde55bb43aea24b0bb354b87f\", \"train.py\": \"cd8f4b4630177e6bff8436956e21bbedf0e47e16f5e02c31057da1ae0a4230b0\", \"viz.py\": \"2f6feda08ffe0d1abf476c291a33e6d86dd0761f021de440967628da0f64714e\"}")
for relative, encoded_content in encoded_files.items():
    destination = SOURCE_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    payload = zlib.decompress(base64.b64decode(encoded_content))
    payload.decode("utf-8")
    observed_hash = hashlib.sha256(payload).hexdigest()
    if observed_hash != expected_hashes[relative]:
        raise RuntimeError(f"Embedded source integrity failure: {relative}")
    destination.write_bytes(payload)
print(f"Extracted {len(encoded_files)} versioned source/config files")


In [ ]:
PRESIGNED_CONFIG_ZLIB_B64 = ''
if PRESIGNED_CONFIG_ZLIB_B64:
    presigned_path = PROJECT_DIR / "s3_presigned_config.json"
    presigned_path.write_bytes(zlib.decompress(base64.b64decode(PRESIGNED_CONFIG_ZLIB_B64)))
    presigned = json.loads(presigned_path.read_text(encoding="utf-8"))
    os.environ["S3_PRESIGNED_CONFIG_PATH"] = str(presigned_path)
    os.environ["S3_BUCKET"] = presigned["bucket"]
    os.environ["S3_PREFIX"] = presigned["s3_prefix"]
    os.environ["RUN_ID"] = presigned["run_id"]
    os.environ["AWS_REGION"] = presigned["aws_region"]
    os.environ["AWS_DEFAULT_REGION"] = presigned["aws_region"]
    print("Loaded short-lived object-scoped S3 operations; no AWS key is embedded.")
else:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    aliases = {
        "AWS_ACCESS_KEY_ID": ("AWS_ACCESS_KEY_ID",),
        "AWS_SECRET_ACCESS_KEY": ("AWS_SECRET_ACCESS_KEY",),
        "AWS_REGION": ("AWS_REGION", "AWS_DEFAULT_REGION"),
        "AWS_DEFAULT_REGION": ("AWS_DEFAULT_REGION", "AWS_REGION"),
        "S3_BUCKET": ("S3_BUCKET",),
        "S3_PREFIX": ("S3_PREFIX",),
    }
    missing = []
    for environment_name, candidates in aliases.items():
        value = None
        for candidate in candidates:
            try:
                value = client.get_secret(candidate)
            except Exception:
                value = None
            if value:
                break
        if value:
            os.environ[environment_name] = value
        else:
            missing.append("/".join(candidates))
    if missing:
        raise RuntimeError("Missing S3 configuration: " + ", ".join(sorted(set(missing))))
os.environ["PYTHONHASHSEED"] = "2026"
print("S3 environment configured; credential values were not printed.")


In [ ]:
required = {
    "lightgbm": "lightgbm>=4.0,<5",
    "boto3": "boto3>=1.34,<2",
    "requests": "requests>=2.31,<3",
}
missing = []
for module, requirement in required.items():
    try:
        imported = __import__(module)
        if module == "lightgbm" and not str(imported.__version__).startswith("4."):
            missing.append(requirement)
    except ImportError:
        missing.append(requirement)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
import lightgbm as lgb
import psutil
host_ram_gib = psutil.virtual_memory().total / (1024 ** 3)
print(f"LightGBM={lgb.__version__}; LightGBM device=CPU; Kaggle accelerator=none; RAM={host_ram_gib:.1f} GiB")


In [ ]:
preferred = Path("/kaggle/input/cicddos2019-parquet")
if preferred.exists():
    data_dir = preferred
else:
    parquet_files = sorted(Path("/kaggle/input").rglob("*.parquet"))
    if not parquet_files:
        raise FileNotFoundError("No Parquet files found in attached Kaggle inputs")
    data_dir = Path(os.path.commonpath([str(path.parent) for path in parquet_files]))
print(f"Preparing deterministic leakage-safe splits from {data_dir}")
data_command = [
    sys.executable, str(SOURCE_DIR / "data.py"),
    "--config", str(SOURCE_DIR / "config/data.smoke.json"),
    "--data-dir", str(data_dir),
    "--output-dir", str(PREPARED_DIR),
]
if False:
    data_command.append("--full-dataset")
if os.environ.get("RUN_ID"):
    data_command.extend([
        "--s3-config", str(SOURCE_DIR / "config/train.smoke.json"),
        "--run-id", os.environ["RUN_ID"],
        "--maximum-hours", "12",
        "--stop-before-minutes", "30",
    ])
data_result = subprocess.run(data_command, cwd=SOURCE_DIR, check=False)
if data_result.returncode not in (0, 75):
    raise subprocess.CalledProcessError(data_result.returncode, data_command)
PREPROCESSING_PAUSED = data_result.returncode == 75
if PREPROCESSING_PAUSED:
    print("Preprocessing paused after a durable source-file checkpoint; training is deferred to the next session.")


In [ ]:
if PREPROCESSING_PAUSED:
    print("Skipping training in this session because preprocessing will resume first.")
else:
    train_command = [
    sys.executable, str(SOURCE_DIR / "train.py"),
    "--config", str(SOURCE_DIR / "config/train.smoke.json"),
    "--prepared-data-dir", str(PREPARED_DIR),
    "--output-dir", str(RUNS_DIR),
    "--upload-checkpoints-to-s3",
    ]

    if os.environ.get("RUN_ID"):
        train_command.extend(["--run-id", os.environ["RUN_ID"]])
    result = subprocess.run(train_command, cwd=SOURCE_DIR, check=False)
    if result.returncode not in (0, 75):
        raise subprocess.CalledProcessError(result.returncode, train_command)
    if result.returncode == 75:
        print("Session paused only after a verified checkpoint; the watchdog may launch the next session.")
    else:
        print("Training reached iteration 100 and final reporting completed or remains durably retryable.")


In [ ]:
active_path = RUNS_DIR / "active_run.json"
if active_path.exists():
    active = json.loads(active_path.read_text(encoding="utf-8"))
    print(json.dumps({
        "run_id": active.get("run_id"),
        "status": active.get("status"),
        "current_iteration": active.get("current_iteration"),
    }, indent=2))
else:
    print("No active run pointer was created.")
